# 03 - RQ3 Human Review Oversight & Intervention Bifurcation Evaluation

### Workflow Architecture
1. **Section 1: Review & Comment Artifact Ingestion and Bot Sanitization Engine (Steps 1.1–1.3)**:
   - Load review and comment artifacts across AIDev v5 and MOSAIC-3M cohorts.
   - Profile actor distribution and automated bot services (GitHub Apps, CI/CD bots, triage automation).
   - Execute regex-based human intervention filtering (bot purge engine) to strictly isolate human reviewer activities.
2. **Section 2: Metric Engineering, Self-Comment Audits, and Baseline Silent Integration (Steps 2.1–2.6)**:
   - Engineer PR-level review metrics: review iterations and discussion comments under Raw and Sanitized bounds.
   - Audit author self-comments and assess reclassification risk across governance tiers.
   - Measure continuous comment volume shift between Raw and Sanitized metrics.
   - Quantify the baseline "Silent Integration" phenomenon (`human_review_comments == 0 & human_review_iterations == 0`).
   - Analyze the merge latency distribution of silent vs. explicitly reviewed PRs.
   - Profile silent integrations across architectural archetypes and AI agent sources.
3. **Section 3: Matched Cohort Linking and Baseline 1 Evaluation (Agentic Infrastructure Friction)**:
   - Left-merge human review metrics onto exact-matched agentic cohorts (`df_aidev_matched`, `df_mosaic_matched`).
   - Evaluate Baseline 1 ATT: assess whether co-changing infrastructure files inflates review iterations or comments for AI agents.
4. **Section 4: Cross-Sectional Human Baseline Equivalence and Intervention Bifurcation (Steps 4.1–4.4)**:
   - Execute Baseline 2 1:1 caliper matching on human developer contributions (`Human App+Infra` vs. `Human App-Only`).
   - Verify cross-sectional baseline equivalence (median file volume parity) between Agent Control and Human Control groups.
   - Calculate Intervention Deltas to isolate the oversight bifurcation between human and agentic contributions.
   - Evaluate paired Silent Integration Matched Odds Ratios using the Exact McNemar Test with 95% Confidence Intervals.
5. **Section 5: Governance Moderation Analysis (Permissions vs. Agent Identity) (Steps 5.1–5.2)**:
   - Estimate pooled interaction logistic regression model predicting external review probability ($\Pr(\text{is\_externally\_reviewed} = 1)$).
   - Execute governance-stratified moderation analysis across Stratum A (External Contributors) and Stratum B (Core Maintainers).
6. **Section 6: Contextual Review Bottlenecks and Review Latency Dynamics (Steps 6.1–6.2)**:
   - Decompose review friction across infrastructure subcategories (`CI_CD`, `Automation`, `Containers`, `Provisioning`).
   - Evaluate empirical latency dynamics of human review oversight versus silent integration.

In [1]:
import os
import sys
import re
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is on sys.path regardless of execution directory
current_dir = os.path.abspath(".")
if os.path.exists(os.path.join(current_dir, "src", "pipeline_utils.py")) or os.path.exists(os.path.join(current_dir, "pipeline_utils.py")):
    sys.path.insert(0, current_dir)
else:
    parent_dir = os.path.abspath("..")
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

import numpy as np
import pandas as pd
from IPython.display import display

# Import modular pipeline utilities
from src.pipeline_utils import (
    ARCHETYPE_MAPPING,
    map_architectural_archetype,
    map_ecosystem,
    assign_pr_scope,
    parse_mosaic_files_array,
    perform_greedy_strata_matching,
    generate_matched_cohorts,
    CACHE_DIR,
    load_dataset_cached,
    filter_temporal_cohort,
    deduplicate_commits,
    parse_file_metadata,
    classify_file_domain,
    APP_DIRS,
)

# Display settings to ensure wide, clean tabular outputs without line-wrapping
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

print("Environment setup and pipeline utilities successfully loaded.")

Environment setup and pipeline utilities successfully loaded.


## Section 1: Review & Comment Artifact Ingestion and Bot Sanitization Engine
### 1.1 Review & Comment Artifact Ingestion

To investigate whether co-changing infrastructure and application files creates additional human review overhead, we extract review iteration records and discussion comment histories across AIDev v5 and MOSAIC-3M cohorts.

We ingest the complete comment artifacts (`pr_comments`, `pr_reviews`, `pr_review_comments`) across both datasets and verify raw record counts prior to bot sanitization.

In [2]:
print("=== 1.1 Loading Review & Comment Artifacts ===")

# 1. AIDev v5: Load Review & Comment tables
df_aidev_reviews = load_dataset_cached(
    hf_url="hf://datasets/hao-li/AIDev-7.6M/pr_reviews.parquet",
    filename="aidev_pr_reviews.parquet"
)
df_aidev_review_comments = load_dataset_cached(
    hf_url="hf://datasets/hao-li/AIDev-7.6M/pr_review_comments.parquet",
    filename="aidev_pr_review_comments.parquet"
)
df_aidev_comments = load_dataset_cached(
    hf_url="hf://datasets/hao-li/AIDev-7.6M/pr_comments.parquet",
    filename="aidev_pr_comments.parquet"
)

print(f"AIDev Reviews Table:         {len(df_aidev_reviews):,} rows")
print(f"AIDev Review Comments Table: {len(df_aidev_review_comments):,} rows")
print(f"AIDev PR Comments Table:     {len(df_aidev_comments):,} rows")

# 2. MOSAIC-3M: Iterate over 6 agent tracks to load Reviews and Comments tables
mosaic_agents = ["Claude", "Codex", "Copilot", "Devin", "Human", "Jules"]
mosaic_review_dfs = []
mosaic_comment_dfs = []

for agent in mosaic_agents:
    r_df = load_dataset_cached(
        hf_url=f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/Reviews/train-00000-of-00001.parquet",
        filename=f"mosaic_reviews_{agent.lower()}.parquet"
    )
    r_df["agent_source"] = agent
    mosaic_review_dfs.append(r_df)
    
    c_df = load_dataset_cached(
        hf_url=f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/Comments/train-00000-of-00001.parquet",
        filename=f"mosaic_comments_{agent.lower()}.parquet"
    )
    c_df["agent_source"] = agent
    mosaic_comment_dfs.append(c_df)

df_mosaic_reviews = pd.concat(mosaic_review_dfs, ignore_index=True)
df_mosaic_comments = pd.concat(mosaic_comment_dfs, ignore_index=True)

print(f"MOSAIC-3M Reviews Table:     {len(df_mosaic_reviews):,} rows across {len(mosaic_agents)} tracks")
print(f"MOSAIC-3M Comments Table:    {len(df_mosaic_comments):,} rows across {len(mosaic_agents)} tracks")


=== 1.1 Loading Review & Comment Artifacts ===


AIDev Reviews Table:         281,170 rows
AIDev Review Comments Table: 289,780 rows
AIDev PR Comments Table:     373,549 rows


MOSAIC-3M Reviews Table:     66,901 rows across 6 tracks
MOSAIC-3M Comments Table:    104,463 rows across 6 tracks


### 1.2 Actor Distribution & Automated Service EDA

In GitHub repositories, automated bots and CI services (e.g., `github-actions[bot]`, `codecov[bot]`, `dependabot[bot]`) frequently post automated status updates, test failures, and triage diagnostics. 

We profile the distribution of comment and review authors to determine the prevalence of automated services versus authentic human maintainers.

In [3]:
print("=== 1.2 Bot & Agent Identification EDA ===")

# 1. AIDev v5 EDA: Concatenate 'user' column across all three tables
aidev_actor_series = pd.concat([
    df_aidev_reviews["user"].dropna(),
    df_aidev_review_comments["user"].dropna(),
    df_aidev_comments["user"].dropna()
], ignore_index=True)

top50_aidev = aidev_actor_series.value_counts().head(50)
print("\n=======================================================")
print("Top 50 Most Frequent Actors: AIDev v5 (Reviews & Comments)")
print("=======================================================")
print(top50_aidev.to_string())

# 2. MOSAIC-3M EDA: Extract login/name from 'author' struct across both tables
def extract_mosaic_author(author):
    if isinstance(author, dict):
        return author.get("login") or author.get("name")
    return None

mosaic_actor_series = pd.concat([
    df_mosaic_reviews["author"].apply(extract_mosaic_author).dropna(),
    df_mosaic_comments["author"].apply(extract_mosaic_author).dropna()
], ignore_index=True)

top50_mosaic = mosaic_actor_series.value_counts().head(50)
print("\n=======================================================")
print("Top 50 Most Frequent Actors: MOSAIC-3M (Reviews & Comments)")
print("=======================================================")
print(top50_mosaic.to_string())


=== 1.2 Bot & Agent Identification EDA ===



Top 50 Most Frequent Actors: AIDev v5 (Reviews & Comments)
user
Copilot                               180011
copilot-swe-agent[bot]                 50787
github-actions[bot]                    49280
coderabbitai[bot]                      43603
copilot-pull-request-reviewer[bot]     30838
devin-ai-integration[bot]              23009
chatgpt-codex-connector[bot]           21904
cursor[bot]                            21653
vercel[bot]                            13224
pelikhan                               12287
gemini-code-assist[bot]                11692
greptile-apps[bot]                     10122
codecov[bot]                            6816
claude[bot]                             6114
netlify[bot]                            5863
github-advanced-security[bot]           5387
google-labs-jules[bot]                  4929
changeset-bot[bot]                      4645
cubic-dev-ai[bot]                       4085
CLAassistant                            3626
cloudflare-workers-and-pages[bot]  

### 1.3 Human Intervention Filtering (Bot Purge Engine)

To prevent automated bot notifications from artificially inflating human review metrics, we execute an aggressive, deterministic regex-based sanitization pass. 

Our bot detection engine filters out any actor matching `[bot]`, common CI/CD services (e.g., `travis`, `circleci`, `sonar`, `renovate`), and internal service accounts, ensuring review iteration counts and discussion comment volumes strictly reflect authentic human reviewer activities.

In [4]:
print("=== 1.3 Human Intervention Filtering (Bot Purge Engine) ===")

import re

# 1. Exact Matches: Short names that are ONLY bots if they are the exact handle.
# This prevents matching human handles like "jean-claude" or "cursor-king".
EXACT_BOTS = ['claude', 'cursor', 'vercel', 'linear', 'sweep', 'jules']

# 2. Safe Substrings: Highly specific tool names. If these appear anywhere in the handle, it's a bot.
# Notice we changed 'devin' to 'devin-ai' to avoid matching humans named Devin.
SAFE_SUBSTRINGS = [
    'copilot', 'coderabbit', 'devin-ai', 'gemini-code-assist', 'qodo', 'sonar',
    'github-actions', 'github-advanced-security', 'netlify', 'cloudflare-workers', 
    'claassistant', 'azure-pipelines', 'graphite-app', 'sourcery-ai', 'amazon-q', 
    'greptile', 'codacy', 'ellipsis', 'openhands', 'codescene', 'liferay', 
    'stackblitz', 'wingetbot', 'pkg-pr-new', 'dependabot', 'renovate', 'supabase', 
    'snyk', 'coveralls', 'houndci', 'semantic-release', 'pullapprove', 'gitleaks', 
    'pre-commit', 'reviewpad', 'codecov', 'chatgpt', 'codex', 'claudeai', 'google-labs-jules'
]

# Combine into a robust regex:
# ^(?:exact|matches)$ | safe_substrings | \[bot\] | -bot | bot-
pattern_str = r'^(?:' + '|'.join(EXACT_BOTS) + r')$|' + \
              r'(?:' + '|'.join(SAFE_SUBSTRINGS) + r')|' + \
              r'\[bot\]|-bot|bot-'

BOT_REGEX = re.compile(pattern_str, re.IGNORECASE)

def is_human(actor_name: str) -> bool:
    if pd.isna(actor_name) or not isinstance(actor_name, str) or not actor_name.strip():
        return False
    # Return True if the regex does NOT find a bot pattern
    return not bool(BOT_REGEX.search(actor_name.strip()))

# Extract login/name from author struct for MOSAIC-3M
df_mosaic_reviews["author_name"] = df_mosaic_reviews["author"].apply(extract_mosaic_author)
df_mosaic_comments["author_name"] = df_mosaic_comments["author"].apply(extract_mosaic_author)

# Filter all 5 tables to create human-only dataframes
df_aidev_reviews_human = df_aidev_reviews[df_aidev_reviews["user"].apply(is_human)].copy()
df_aidev_review_comments_human = df_aidev_review_comments[df_aidev_review_comments["user"].apply(is_human)].copy()
df_aidev_comments_human = df_aidev_comments[df_aidev_comments["user"].apply(is_human)].copy()

df_mosaic_reviews_human = df_mosaic_reviews[df_mosaic_reviews["author_name"].apply(is_human)].copy()
df_mosaic_comments_human = df_mosaic_comments[df_mosaic_comments["author_name"].apply(is_human)].copy()

# Before-and-after summary table
table_comparisons = [
    ("AIDev Reviews", len(df_aidev_reviews), len(df_aidev_reviews_human)),
    ("AIDev Review Comments", len(df_aidev_review_comments), len(df_aidev_review_comments_human)),
    ("AIDev PR Comments", len(df_aidev_comments), len(df_aidev_comments_human)),
    ("MOSAIC-3M Reviews", len(df_mosaic_reviews), len(df_mosaic_reviews_human)),
    ("MOSAIC-3M Comments", len(df_mosaic_comments), len(df_mosaic_comments_human)),
]

filter_summary = pd.DataFrame([
    {
        "Table": name,
        "Total (Before)": f"{before:,}",
        "Human Only (After)": f"{after:,}",
        "Bots Purged": f"{before - after:,}",
        "Purged (%)": f"{((before - after) / before * 100):.2f}%"
    }
    for name, before, after in table_comparisons
])

print("\n--- Summary: Automated Bot / Agent Purge Across All Review & Comment Tables ---")
print(filter_summary.to_string(index=False))


=== 1.3 Human Intervention Filtering (Bot Purge Engine) ===



--- Summary: Automated Bot / Agent Purge Across All Review & Comment Tables ---
                Table Total (Before) Human Only (After) Bots Purged Purged (%)
        AIDev Reviews        281,170            153,028     128,142     45.57%
AIDev Review Comments        289,780            111,014     178,766     61.69%
    AIDev PR Comments        373,549            135,235     238,314     63.80%
    MOSAIC-3M Reviews         66,901             41,397      25,504     38.12%
   MOSAIC-3M Comments        104,463             33,473      70,990     67.96%


## Section 2: Metric Engineering, Self-Comment Audits, and Baseline Silent Integration
### 2.1 PR-Level Review & Comment Metric Engineering

Following the sanitization pass, we aggregate review activities to the pull request level. For each PR, we compute two complementary dimensions of human intervention:
1. **Human Review Iterations (`human_review_iterations`)**: The number of explicit review events submitted by human maintainers (distinct review approvals, changes requested, or dismissals).
2. **Human Review Comments (`human_review_comments`)**: The total number of discussion comments posted by human actors across issue comments and inline diff comments.

We compute these metrics across both datasets and report descriptive statistics for both treatment (`App_and_Infra`) and control (`App_Only`) cohorts.

In [5]:
print("=== 2.1 Metric Engineering & PR Aggregation ===")

# --------------------------------------------------------------------------
# 1. AIDev v5 Aggregation
# --------------------------------------------------------------------------
# A. Human review iterations (grouped by pr_id from reviews table)
human_review_iterations_aidev = df_aidev_reviews_human.groupby("pr_id")["id"].count().rename("human_review_iterations")

# B. General PR comments
pr_comments_count_aidev = df_aidev_comments_human.groupby("pr_id")["id"].count()

# C. Inline review comments (map pr_id via pull_request_review_id -> df_aidev_reviews.id)
df_rc_mapped = df_aidev_review_comments_human.merge(
    df_aidev_reviews[["id", "pr_id"]].rename(columns={"id": "review_obj_id"}),
    left_on="pull_request_review_id",
    right_on="review_obj_id",
    how="inner"
)
review_comments_count_aidev = df_rc_mapped.groupby("pr_id")["id"].count()

# D. Combine PR comments and inline review comments into total human_review_comments
human_review_comments_aidev = pr_comments_count_aidev.add(review_comments_count_aidev, fill_value=0).astype(int).rename("human_review_comments")

# E. Combine into a single aidev_human_metrics DataFrame
aidev_human_metrics = pd.concat([human_review_iterations_aidev, human_review_comments_aidev], axis=1).fillna(0).astype(int).reset_index()

# --------------------------------------------------------------------------
# 2. MOSAIC-3M Aggregation
# --------------------------------------------------------------------------
human_review_iterations_mosaic = df_mosaic_reviews_human.groupby("pr_id")["id"].count().rename("human_review_iterations")
human_review_comments_mosaic = df_mosaic_comments_human.groupby("pr_id")["id"].count().rename("human_review_comments")
mosaic_human_metrics = pd.concat([human_review_iterations_mosaic, human_review_comments_mosaic], axis=1).fillna(0).astype(int).reset_index()

# --------------------------------------------------------------------------
# 3. Prepare Base Cohorts & Generate Matched PSM Pairs
# --------------------------------------------------------------------------
# Base PR & Repo Ingestion
df_aidev_pr = load_dataset_cached("hf://datasets/hao-li/AIDev-7.6M/pull_request.parquet", "aidev_v5_pull_request.parquet")
df_aidev_pr_filtered = filter_temporal_cohort(df_aidev_pr)
df_aidev_repo = load_dataset_cached("hf://datasets/hao-li/AIDev-7.6M/repository.parquet", "aidev_v5_repository.parquet")
repo_meta_aidev = df_aidev_repo[["id", "language", "stars"]].rename(columns={"id": "repo_id", "language": "primary_language"})
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(repo_meta_aidev, on="repo_id", how="left")
df_aidev_pr_filtered["primary_language"] = df_aidev_pr_filtered["primary_language"].fillna("Unknown")
df_aidev_pr_filtered["stars"] = df_aidev_pr_filtered["stars"].fillna(0)

repo_dfs, pr_dfs, commit_dfs = [], [], []
for agent in mosaic_agents:
    repo_dfs.append(load_dataset_cached(f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/Repositories/train-00000-of-00001.parquet", f"mosaic_repo_{agent.lower()}.parquet"))
    p_df = load_dataset_cached(f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/PullRequests/train-00000-of-00001.parquet", f"mosaic_pr_{agent.lower()}.parquet")
    p_df["agent_source"] = agent
    pr_dfs.append(p_df)
    commit_dfs.append(load_dataset_cached(f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/Commits/train-00000-of-00001.parquet", f"mosaic_commits_{agent.lower()}.parquet", columns=["pr_id", "sha", "additions", "deletions", "message_headline"]))
df_mosaic_repo = pd.concat(repo_dfs, ignore_index=True)
df_mosaic_pr = pd.concat(pr_dfs, ignore_index=True)
df_mosaic_commits = pd.concat(commit_dfs, ignore_index=True)

dedup_key = "name_with_owner" if "name_with_owner" in df_mosaic_repo.columns else "id"
df_mosaic_repo_unique = df_mosaic_repo.drop_duplicates(subset=[dedup_key]).copy()
df_mosaic_repo_gt100 = df_mosaic_repo_unique[df_mosaic_repo_unique["stargazer_count"] > 100].copy()
valid_mosaic_repo_ids = set(df_mosaic_repo_gt100["id"].dropna().unique())

df_mosaic_pr_temporal = filter_temporal_cohort(df_mosaic_pr)
df_mosaic_pr_temporal["base_repo_id"] = df_mosaic_pr_temporal["base_repository"].apply(lambda x: x.get("id") if isinstance(x, dict) else None)
df_mosaic_pr_filtered = df_mosaic_pr_temporal[df_mosaic_pr_temporal["base_repo_id"].isin(valid_mosaic_repo_ids)].copy()
repo_meta_mosaic = df_mosaic_repo_gt100[["id", "primary_language", "stargazer_count"]].rename(columns={"id": "base_repo_id", "stargazer_count": "stars"}).drop_duplicates(subset=["base_repo_id"])
df_mosaic_pr_filtered = df_mosaic_pr_filtered.merge(repo_meta_mosaic, on="base_repo_id", how="left")
df_mosaic_pr_filtered["primary_language"] = df_mosaic_pr_filtered["primary_language"].fillna("Unknown")
df_mosaic_pr_filtered["stars"] = df_mosaic_pr_filtered["stars"].fillna(0)

# Feature extraction & Triad Controls
df_commit_details = load_dataset_cached("hf://datasets/hao-li/AIDev-7.6M/pr_commit_details.parquet", "aidev_v5_pr_commit_details.parquet", columns=["pr_id", "sha", "filename", "status", "changes", "message"])
df_commit_joined = df_commit_details.merge(df_aidev_pr_filtered[["id"]], left_on="pr_id", right_on="id", how="inner")
file_classes = [classify_file_domain(fn, st, ch) for fn, st, ch in zip(df_commit_joined["filename"], df_commit_joined["status"], df_commit_joined["changes"])]
df_commit_joined["is_app"] = [c["is_app"] for c in file_classes]

aidev_total = df_commit_joined.dropna(subset=["filename"]).groupby("pr_id")["filename"].nunique().reset_index(name="total_files_recorded")
aidev_app = df_commit_joined[df_commit_joined["is_app"]].dropna(subset=["filename"]).groupby("pr_id")["filename"].nunique().reset_index(name="app_count")
aidev_app_churn = df_commit_joined[df_commit_joined["is_app"]].groupby("pr_id")["changes"].sum().reset_index(name="app_line_churn")

aidev_files = aidev_total.merge(aidev_app, on="pr_id", how="left").merge(aidev_app_churn, on="pr_id", how="left")
aidev_files["app_count"] = aidev_files["app_count"].fillna(0).astype(int)
aidev_files["app_line_churn"] = aidev_files["app_line_churn"].fillna(0).astype(int)

df_aidev_pr_filtered = df_aidev_pr_filtered.merge(aidev_files, left_on="id", right_on="pr_id", how="left")
df_aidev_pr_filtered["total_files_recorded"] = df_aidev_pr_filtered["total_files_recorded"].fillna(0).astype(int)
df_aidev_pr_filtered["app_count"] = df_aidev_pr_filtered["app_count"].fillna(0).astype(int)
df_aidev_pr_filtered["app_line_churn"] = df_aidev_pr_filtered["app_line_churn"].fillna(0).astype(int)

mosaic_file_metrics = df_mosaic_pr_filtered["files"].apply(parse_mosaic_files_array)
df_mosaic_pr_filtered["total_files_recorded"] = mosaic_file_metrics["total_files_recorded"].astype(int)
df_mosaic_pr_filtered["app_count"] = mosaic_file_metrics["app_count"].astype(int)
df_mosaic_pr_filtered["infra_count"] = mosaic_file_metrics["infra_count"].astype(int)
df_mosaic_pr_filtered["app_line_churn"] = mosaic_file_metrics["app_line_churn"].astype(int)
df_mosaic_pr_filtered["infra_line_churn"] = mosaic_file_metrics["infra_line_churn"].astype(int)
df_mosaic_pr_filtered["has_tests"] = mosaic_file_metrics["has_tests"]

aidev_pre_drop = len(df_aidev_pr_filtered)
mosaic_pre_drop = len(df_mosaic_pr_filtered)

df_aidev_pr_filtered = df_aidev_pr_filtered[df_aidev_pr_filtered["app_count"] > 0].copy()
df_mosaic_pr_filtered = df_mosaic_pr_filtered[df_mosaic_pr_filtered["app_count"] > 0].copy()

df_commit_details["seq_order"] = df_commit_details.index
df_commit_joined = df_commit_details.merge(df_aidev_pr_filtered[["id"]], left_on="pr_id", right_on="id", how="inner")
df_commit_dedup = deduplicate_commits(df_commit_joined)

file_meta = df_commit_dedup["filename"].apply(parse_file_metadata)
df_commit_dedup["root_dir"] = file_meta.apply(lambda x: x["root_dir"])
file_classes_dedup = [classify_file_domain(fn, st, ch) for fn, st, ch in zip(df_commit_dedup["filename"], df_commit_dedup["status"], df_commit_dedup["changes"])]
df_commit_dedup["is_app"] = [c["is_app"] for c in file_classes_dedup]
df_commit_dedup["is_infra"] = [c["is_infra"] for c in file_classes_dedup]
df_commit_dedup["sub_type"] = [c["sub_type"] for c in file_classes_dedup]

pr_scope = df_commit_dedup.groupby("pr_id").agg(app_files=('is_app', 'sum'), infra_files=('is_infra', 'sum')).reset_index()
pr_scope["scope"] = pr_scope.apply(lambda x: assign_pr_scope(x['app_files'], x['infra_files']), axis=1)

aidev_test_count = df_commit_dedup[df_commit_dedup["sub_type"] == "Test"].groupby("pr_id").size().reset_index(name="test_count")
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(aidev_test_count, left_on="id", right_on="pr_id", how="left")
df_aidev_pr_filtered["test_count"] = df_aidev_pr_filtered["test_count"].fillna(0).astype(int)
df_aidev_pr_filtered['has_tests'] = df_aidev_pr_filtered['test_count'] > 0

if "pr_id" in df_aidev_pr_filtered.columns:
    df_aidev_pr_filtered.drop(columns=["pr_id"], inplace=True)
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(pr_scope[["pr_id", "scope"]], left_on="id", right_on="pr_id", how="left")
if "pr_id" in df_aidev_pr_filtered.columns:
    df_aidev_pr_filtered.drop(columns=["pr_id"], inplace=True)
df_aidev_pr_filtered["scope"] = df_aidev_pr_filtered["scope"].fillna("Other")

df_app_commits = df_commit_dedup[df_commit_dedup["is_app"] == True].copy()
root_counts = df_app_commits.groupby(["pr_id", "root_dir"]).size().reset_index(name="file_count")
max_counts = root_counts.groupby("pr_id")["file_count"].transform("max")
top_roots = root_counts[root_counts["file_count"] == max_counts]
tie_counts = top_roots.groupby("pr_id").size()
tied_pr_ids = set(tie_counts[tie_counts > 1].index)
primary_roots = top_roots.drop_duplicates(subset=["pr_id"], keep="first").copy()
primary_roots.loc[primary_roots["pr_id"].isin(tied_pr_ids), "root_dir"] = "Cross-Domain"
primary_roots.rename(columns={"root_dir": "primary_root_dir"}, inplace=True)
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(primary_roots[["pr_id", "primary_root_dir"]], left_on="id", right_on="pr_id", how="left")
if "pr_id" in df_aidev_pr_filtered.columns:
    df_aidev_pr_filtered.drop(columns=["pr_id"], inplace=True)
df_aidev_pr_filtered["primary_root_dir"] = df_aidev_pr_filtered["primary_root_dir"].fillna("(none)")

mosaic_meta = df_mosaic_pr_filtered["files"].apply(parse_mosaic_files_array)
df_mosaic_pr_filtered["scope"] = mosaic_meta.apply(lambda x: assign_pr_scope(x['app_count'], x['infra_count']), axis=1)
df_mosaic_pr_filtered["primary_root_dir"] = mosaic_meta["primary_root_dir"]

df_aidev_pr_filtered["architectural_archetype"] = df_aidev_pr_filtered["primary_root_dir"].map(map_architectural_archetype)
df_mosaic_pr_filtered["architectural_archetype"] = df_mosaic_pr_filtered["primary_root_dir"].map(map_architectural_archetype)

# --------------------------------------------------------------------------

# --------------------------------------------------------------------------
# 4. Diagnostic Logging: Review Linkage, Scope Census, & Archetypes
# --------------------------------------------------------------------------
_temp_aidev = df_aidev_pr_filtered[['id']].merge(aidev_human_metrics, left_on='id', right_on='pr_id', how='left').fillna(0)
_temp_mosaic = df_mosaic_pr_filtered[['id']].merge(mosaic_human_metrics, left_on='id', right_on='pr_id', how='left').fillna(0)

# Linkage counts
aidev_retained = len(df_aidev_pr_filtered)
aidev_iter_gt0 = (_temp_aidev['human_review_iterations'] > 0).sum()
aidev_comm_gt0 = (_temp_aidev['human_review_comments'] > 0).sum()
aidev_zero_act = ((_temp_aidev['human_review_iterations'] == 0) & (_temp_aidev['human_review_comments'] == 0)).sum()

mosaic_retained = len(df_mosaic_pr_filtered)
mosaic_iter_gt0 = (_temp_mosaic['human_review_iterations'] > 0).sum()
mosaic_comm_gt0 = (_temp_mosaic['human_review_comments'] > 0).sum()
mosaic_zero_act = ((_temp_mosaic['human_review_iterations'] == 0) & (_temp_mosaic['human_review_comments'] == 0)).sum()

# Scope census
aidev_scope_counts = df_aidev_pr_filtered['scope'].value_counts()
mosaic_agentic_scope = df_mosaic_pr_filtered[df_mosaic_pr_filtered['agent_source'] != 'Human']['scope'].value_counts()
mosaic_human_scope = df_mosaic_pr_filtered[df_mosaic_pr_filtered['agent_source'] == 'Human']['scope'].value_counts()
tot_m_agent = len(df_mosaic_pr_filtered[df_mosaic_pr_filtered['agent_source'] != 'Human'])
tot_m_human = len(df_mosaic_pr_filtered[df_mosaic_pr_filtered['agent_source'] == 'Human'])

print("\n--- A. Attrition Funnel (Data Cleaning Log) ---")
print(f"AIDev v5:   Initial PRs: {aidev_pre_drop:,} -> Dropped (app_count==0): {aidev_pre_drop - aidev_retained:,} ({(aidev_pre_drop - aidev_retained)/aidev_pre_drop*100:.2f}%) -> Retained: {aidev_retained:,}")
print(f"MOSAIC-3M:  Initial PRs: {mosaic_pre_drop:,} -> Dropped (app_count==0): {mosaic_pre_drop - mosaic_retained:,} ({(mosaic_pre_drop - mosaic_retained)/mosaic_pre_drop*100:.2f}%) -> Retained: {mosaic_retained:,}")

print("\n--- B. Human Review Linkage & Coverage (Retained Cohort) ---")
print(f"AIDev v5 (Total: {aidev_retained:,}):")
print(f"  - PRs with >= 1 human review iteration: {aidev_iter_gt0:,} ({aidev_iter_gt0/aidev_retained*100:.2f}%)")
print(f"  - PRs with >= 1 human comment:          {aidev_comm_gt0:,} ({aidev_comm_gt0/aidev_retained*100:.2f}%)")
print(f"  - PRs with 0 recorded human activity:   {aidev_zero_act:,} ({aidev_zero_act/aidev_retained*100:.2f}%)")
print(f"MOSAIC-3M (Total: {mosaic_retained:,}):")
print(f"  - PRs with >= 1 human review iteration: {mosaic_iter_gt0:,} ({mosaic_iter_gt0/mosaic_retained*100:.2f}%)")
print(f"  - PRs with >= 1 human comment:          {mosaic_comm_gt0:,} ({mosaic_comm_gt0/mosaic_retained*100:.2f}%)")
print(f"  - PRs with 0 recorded human activity:   {mosaic_zero_act:,} ({mosaic_zero_act/mosaic_retained*100:.2f}%)")
print("Join Integrity Check Passed: 0 NaNs in review metrics.")

print("\n--- C. Pre-Match Scope Census (Baseline Denominator) ---")
print(f"AIDev v5 (Agentic, Total: {aidev_retained:,}):")
for s in ['App_Only', 'App_and_Infra']:
    cnt = aidev_scope_counts.get(s, 0)
    print(f"  - {s:<14} {cnt:,} ({cnt/aidev_retained*100:.2f}%)")
print(f"MOSAIC-3M Agentic (Total: {tot_m_agent:,}):")
for s in ['App_Only', 'App_and_Infra']:
    cnt = mosaic_agentic_scope.get(s, 0)
    print(f"  - {s:<14} {cnt:,} ({cnt/tot_m_agent*100:.2f}%)")
print(f"MOSAIC-3M Human Baseline (Total: {tot_m_human:,}):")
for s in ['App_Only', 'App_and_Infra']:
    cnt = mosaic_human_scope.get(s, 0)
    print(f"  - {s:<14} {cnt:,} ({cnt/tot_m_human*100:.2f}%)")

print("\n--- D. Architectural Archetype Distribution ---")
print("AIDev v5:")
for arch, cnt in df_aidev_pr_filtered['architectural_archetype'].value_counts().head(5).items():
    print(f"  - {arch:<25} {cnt:,} ({cnt/aidev_retained*100:.2f}%)")
print("MOSAIC-3M:")
for arch, cnt in df_mosaic_pr_filtered['architectural_archetype'].value_counts().head(5).items():
    print(f"  - {arch:<25} {cnt:,} ({cnt/mosaic_retained*100:.2f}%)")


=== 2.1 Metric Engineering & PR Aggregation ===



--- A. Attrition Funnel (Data Cleaning Log) ---
AIDev v5:   Initial PRs: 26,870 -> Dropped (app_count==0): 12,123 (45.12%) -> Retained: 14,747
MOSAIC-3M:  Initial PRs: 5,206 -> Dropped (app_count==0): 1,558 (29.93%) -> Retained: 3,648

--- B. Human Review Linkage & Coverage (Retained Cohort) ---
AIDev v5 (Total: 14,747):
  - PRs with >= 1 human review iteration: 2,951 (20.01%)
  - PRs with >= 1 human comment:          2,353 (15.96%)
  - PRs with 0 recorded human activity:   11,323 (76.78%)
MOSAIC-3M (Total: 3,648):
  - PRs with >= 1 human review iteration: 2,083 (57.10%)
  - PRs with >= 1 human comment:          1,163 (31.88%)
  - PRs with 0 recorded human activity:   1,241 (34.02%)
Join Integrity Check Passed: 0 NaNs in review metrics.

--- C. Pre-Match Scope Census (Baseline Denominator) ---
AIDev v5 (Agentic, Total: 14,747):
  - App_Only       13,768 (93.36%)
  - App_and_Infra  979 (6.64%)
MOSAIC-3M Agentic (Total: 1,920):
  - App_Only       1,799 (93.70%)
  - App_and_Infra  121 (6

### 2.2 Author Self-Comment Audit & Reclassification Risk Analysis

A critical threat to construct validity in pull request review analytics is **Author Self-Commenting**: PR authors often comment on their own PRs to provide implementation context, respond to CI failures, or document architectural decisions. If author self-comments are included in discussion counts, a PR with zero maintainer interaction might be incorrectly classified as having received explicit human review.

To evaluate this construct validity risk, we isolate PR author self-comments from external maintainer comments:
1. We compute `author_self_comments` versus `external_comments` across all PRs.
2. We identify PRs where all comments were authored solely by the PR author (`author_self_comments > 0 & external_comments == 0`).
3. We audit the reclassification rate across governance tiers (`CONTRIBUTOR`, `NONE`, `MEMBER`, `COLLABORATOR`, `OWNER`) to quantify how many silently integrated PRs would be misclassified under raw comment counts.

In [6]:
print("=== 2.2 Diagnostic: PR Author Self-Comment Audit & Reclassification Risk ===")

# --------------------------------------------------------------------------
# 1. Null-Safe Author Extraction & Normalization
# --------------------------------------------------------------------------
clean_handle = lambda x: str(x).lower().strip() if pd.notna(x) and str(x).strip().lower() not in ['none', 'nan', ''] else None

# AIDev v5: PR author is in 'user'
df_aidev_pr_filtered["pr_author_norm"] = df_aidev_pr_filtered["user"].apply(clean_handle)
aidev_pr_author_map = dict(zip(df_aidev_pr_filtered["id"], df_aidev_pr_filtered["pr_author_norm"]))

# MOSAIC-3M: PR author is extracted from 'author' struct
df_mosaic_pr_filtered["pr_author_name"] = df_mosaic_pr_filtered["author"].apply(extract_mosaic_author)
df_mosaic_pr_filtered["pr_author_norm"] = df_mosaic_pr_filtered["pr_author_name"].apply(clean_handle)
mosaic_pr_author_map = dict(zip(df_mosaic_pr_filtered["id"], df_mosaic_pr_filtered["pr_author_norm"]))
mosaic_pr_cohort_map = dict(zip(
    df_mosaic_pr_filtered["id"], 
    df_mosaic_pr_filtered["agent_source"].apply(lambda x: "MOSAIC-3M (Human Baseline)" if x == "Human" else "MOSAIC-3M (Agentic Only)")
))

# --------------------------------------------------------------------------
# 2. Link Comments & Formal Reviews to PR Authors
# --------------------------------------------------------------------------
# A. AIDev General PR Comments
df_aidev_comments_diag = df_aidev_comments_human.copy()
df_aidev_comments_diag["commenter_norm"] = df_aidev_comments_diag["user"].apply(clean_handle)
df_aidev_comments_diag["pr_author_norm"] = df_aidev_comments_diag["pr_id"].map(aidev_pr_author_map)
df_aidev_comments_diag["is_author_comment"] = (df_aidev_comments_diag["pr_author_norm".notna()]) if False else ((df_aidev_comments_diag["pr_author_norm"].notna()) & (df_aidev_comments_diag["commenter_norm"].notna()) & (df_aidev_comments_diag["commenter_norm"] == df_aidev_comments_diag["pr_author_norm"]))
df_aidev_comments_diag["is_external_comment"] = (df_aidev_comments_diag["pr_author_norm"].notna()) & (df_aidev_comments_diag["commenter_norm"].notna()) & (df_aidev_comments_diag["commenter_norm"] != df_aidev_comments_diag["pr_author_norm"])

# B. AIDev Inline Review Comments (mapped via pull_request_review_id -> df_aidev_reviews.pr_id)
df_rc_diag = df_rc_mapped.copy()
df_rc_diag["commenter_norm"] = df_rc_diag["user"].apply(clean_handle)
df_rc_diag["pr_author_norm"] = df_rc_diag["pr_id"].map(aidev_pr_author_map)
df_rc_diag["is_author_comment"] = (df_rc_diag["pr_author_norm"].notna()) & (df_rc_diag["commenter_norm"].notna()) & (df_rc_diag["commenter_norm"] == df_rc_diag["pr_author_norm"])
df_rc_diag["is_external_comment"] = (df_rc_diag["pr_author_norm"].notna()) & (df_rc_diag["commenter_norm"].notna()) & (df_rc_diag["commenter_norm"] != df_rc_diag["pr_author_norm"])

# Formal Reviews Sanity Check & External Iterations (AIDev v5)
df_aidev_reviews_diag = df_aidev_reviews_human.copy()
df_aidev_reviews_diag["reviewer_norm"] = df_aidev_reviews_diag["user"].apply(clean_handle)
df_aidev_reviews_diag["pr_author_norm"] = df_aidev_reviews_diag["pr_id"].map(aidev_pr_author_map)
df_aidev_reviews_diag["is_author_review"] = (df_aidev_reviews_diag["pr_author_norm"].notna()) & (df_aidev_reviews_diag["reviewer_norm"].notna()) & (df_aidev_reviews_diag["reviewer_norm"] == df_aidev_reviews_diag["pr_author_norm"])
df_aidev_reviews_diag["is_external_review"] = (df_aidev_reviews_diag["pr_author_norm"].notna()) & (df_aidev_reviews_diag["reviewer_norm"].notna()) & (df_aidev_reviews_diag["reviewer_norm"] != df_aidev_reviews_diag["pr_author_norm"])

# C. MOSAIC-3M Comments
df_mosaic_comments_diag = df_mosaic_comments_human.copy()
df_mosaic_comments_diag["commenter_norm"] = df_mosaic_comments_diag["author_name"].apply(clean_handle)
df_mosaic_comments_diag["pr_author_norm"] = df_mosaic_comments_diag["pr_id"].map(mosaic_pr_author_map)
df_mosaic_comments_diag["cohort"] = df_mosaic_comments_diag["pr_id"].map(mosaic_pr_cohort_map)
df_mosaic_comments_diag["is_author_comment"] = (df_mosaic_comments_diag["pr_author_norm"].notna()) & (df_mosaic_comments_diag["commenter_norm"].notna()) & (df_mosaic_comments_diag["commenter_norm"] == df_mosaic_comments_diag["pr_author_norm"])
df_mosaic_comments_diag["is_external_comment"] = (df_mosaic_comments_diag["pr_author_norm"].notna()) & (df_mosaic_comments_diag["commenter_norm"].notna()) & (df_mosaic_comments_diag["commenter_norm"] != df_mosaic_comments_diag["pr_author_norm"])

# Formal Reviews Sanity Check & External Iterations (MOSAIC-3M)
df_mosaic_reviews_diag = df_mosaic_reviews_human.copy()
df_mosaic_reviews_diag["reviewer_norm"] = df_mosaic_reviews_diag["author_name"].apply(clean_handle)
df_mosaic_reviews_diag["pr_author_norm"] = df_mosaic_reviews_diag["pr_id"].map(mosaic_pr_author_map)
df_mosaic_reviews_diag["cohort"] = df_mosaic_reviews_diag["pr_id"].map(mosaic_pr_cohort_map)
df_mosaic_reviews_diag["is_author_review"] = (df_mosaic_reviews_diag["pr_author_norm"].notna()) & (df_mosaic_reviews_diag["reviewer_norm"].notna()) & (df_mosaic_reviews_diag["reviewer_norm"] == df_mosaic_reviews_diag["pr_author_norm"])
df_mosaic_reviews_diag["is_external_review"] = (df_mosaic_reviews_diag["pr_author_norm"].notna()) & (df_mosaic_reviews_diag["reviewer_norm"].notna()) & (df_mosaic_reviews_diag["reviewer_norm"] != df_mosaic_reviews_diag["pr_author_norm"])

print("--- Sanity Check: Formal Review Events where Reviewer == PR Author ---")
aidev_rev_cohort = df_aidev_reviews_diag[df_aidev_reviews_diag["pr_author_norm"].notna()]
aidev_rev_count = len(aidev_rev_cohort)
aidev_rev_self = aidev_rev_cohort["is_author_review"].sum()
print(f"AIDev v5 (Agentic):          {aidev_rev_self:,} / {aidev_rev_count:,} reviews ({aidev_rev_self / aidev_rev_count * 100:.2f}%)")

mosaic_rev_cohorts = df_mosaic_reviews_diag[df_mosaic_reviews_diag["cohort"].notna()]
for c in ["MOSAIC-3M (Agentic Only)", "MOSAIC-3M (Human Baseline)"]:
    sub = mosaic_rev_cohorts[mosaic_rev_cohorts["cohort"] == c]
    tot = len(sub)
    self_n = sub["is_author_review"].sum()
    pct = (self_n / tot * 100) if tot > 0 else 0.0
    print(f"{c:<28} {self_n:,} / {tot:,} reviews ({pct:.2f}%)")

# --------------------------------------------------------------------------
# 3. PR-Level Comment Aggregations & Sanitized Metrics
# --------------------------------------------------------------------------
# AIDev PR Aggregations
aidev_gen_author = df_aidev_comments_diag[df_aidev_comments_diag["is_author_comment"]].groupby("pr_id").size().rename("author_gen_comments")
aidev_gen_ext = df_aidev_comments_diag[df_aidev_comments_diag["is_external_comment"]].groupby("pr_id").size().rename("ext_gen_comments")

aidev_inline_author = df_rc_diag[df_rc_diag["is_author_comment"]].groupby("pr_id").size().rename("author_inline_comments")
aidev_inline_ext = df_rc_diag[df_rc_diag["is_external_comment"]].groupby("pr_id").size().rename("ext_inline_comments")

ext_rev_aidev = df_aidev_reviews_diag[df_aidev_reviews_diag["is_external_review"]].groupby("pr_id").size().rename("external_review_iterations")

df_eval_aidev = df_aidev_pr_filtered[["id", "scope", "pr_author_norm"]].copy()
df_eval_aidev = df_eval_aidev.merge(aidev_human_metrics[["pr_id", "human_review_iterations", "human_review_comments"]], left_on="id", right_on="pr_id", how="left")
df_eval_aidev["human_review_iterations"] = df_eval_aidev["human_review_iterations"].fillna(0).astype(int)
df_eval_aidev["human_review_comments"] = df_eval_aidev["human_review_comments"].fillna(0).astype(int)

df_eval_aidev = df_eval_aidev.merge(aidev_gen_author, left_on="id", right_index=True, how="left").merge(aidev_gen_ext, left_on="id", right_index=True, how="left")
df_eval_aidev = df_eval_aidev.merge(aidev_inline_author, left_on="id", right_index=True, how="left").merge(aidev_inline_ext, left_on="id", right_index=True, how="left")
df_eval_aidev = df_eval_aidev.merge(ext_rev_aidev, left_on="id", right_index=True, how="left")

df_eval_aidev[["author_gen_comments", "ext_gen_comments", "author_inline_comments", "ext_inline_comments", "external_review_iterations"]] = df_eval_aidev[["author_gen_comments", "ext_gen_comments", "author_inline_comments", "ext_inline_comments", "external_review_iterations"]].fillna(0).astype(int)

df_eval_aidev["author_comments"] = df_eval_aidev["author_gen_comments"] + df_eval_aidev["author_inline_comments"]
df_eval_aidev["external_comments"] = df_eval_aidev["ext_gen_comments"] + df_eval_aidev["ext_inline_comments"]
df_eval_aidev["total_comments"] = df_eval_aidev["author_comments"] + df_eval_aidev["external_comments"]

df_eval_aidev["current_silent"] = (df_eval_aidev["human_review_comments"] == 0) & (df_eval_aidev["human_review_iterations"] == 0)
df_eval_aidev["current_reviewed"] = ~df_eval_aidev["current_silent"]
df_eval_aidev["sanitized_silent"] = (df_eval_aidev["external_comments"] == 0) & (df_eval_aidev["external_review_iterations"] == 0)
df_eval_aidev["at_risk_flip"] = (df_eval_aidev["current_reviewed"] == True) & (df_eval_aidev["sanitized_silent"] == True)

# MOSAIC PR Aggregations
mosaic_author_c = df_mosaic_comments_diag[df_mosaic_comments_diag["is_author_comment"]].groupby("pr_id").size().rename("author_comments")
mosaic_ext_c = df_mosaic_comments_diag[df_mosaic_comments_diag["is_external_comment"]].groupby("pr_id").size().rename("ext_comments")
ext_rev_mosaic = df_mosaic_reviews_diag[df_mosaic_reviews_diag["is_external_review"]].groupby("pr_id").size().rename("external_review_iterations")

df_eval_mosaic = df_mosaic_pr_filtered[["id", "scope", "agent_source", "author_association", "pr_author_norm"]].copy()
df_eval_mosaic["cohort"] = df_eval_mosaic["agent_source"].apply(lambda x: "MOSAIC-3M (Human Baseline)" if x == "Human" else "MOSAIC-3M (Agentic Only)")
df_eval_mosaic = df_eval_mosaic.merge(mosaic_human_metrics[["pr_id", "human_review_iterations", "human_review_comments"]], left_on="id", right_on="pr_id", how="left")
df_eval_mosaic["human_review_iterations"] = df_eval_mosaic["human_review_iterations"].fillna(0).astype(int)
df_eval_mosaic["human_review_comments"] = df_eval_mosaic["human_review_comments"].fillna(0).astype(int)

df_eval_mosaic = df_eval_mosaic.merge(mosaic_author_c, left_on="id", right_index=True, how="left").merge(mosaic_ext_c, left_on="id", right_index=True, how="left")
df_eval_mosaic = df_eval_mosaic.merge(ext_rev_mosaic, left_on="id", right_index=True, how="left")
df_eval_mosaic[["author_comments", "ext_comments", "external_review_iterations"]] = df_eval_mosaic[["author_comments", "ext_comments", "external_review_iterations"]].fillna(0).astype(int)
df_eval_mosaic["external_comments"] = df_eval_mosaic["ext_comments"]
df_eval_mosaic["total_comments"] = df_eval_mosaic["author_comments"] + df_eval_mosaic["external_comments"]

df_eval_mosaic["current_silent"] = (df_eval_mosaic["human_review_comments"] == 0) & (df_eval_mosaic["human_review_iterations"] == 0)
df_eval_mosaic["current_reviewed"] = ~df_eval_mosaic["current_silent"]
df_eval_mosaic["sanitized_silent"] = (df_eval_mosaic["external_comments"] == 0) & (df_eval_mosaic["external_review_iterations"] == 0)
df_eval_mosaic["at_risk_flip"] = (df_eval_mosaic["current_reviewed"] == True) & (df_eval_mosaic["sanitized_silent"] == True)

# --------------------------------------------------------------------------
# 4. Table 1: Comment Volume & Authorship Audit
# --------------------------------------------------------------------------
print("\n--- Table 1: Comment Volume & Authorship Audit ---")
t1_rows = []

aidev_tot_c = df_eval_aidev["total_comments"].sum()
aidev_auth_c = df_eval_aidev["author_comments"].sum()
aidev_ext_c = df_eval_aidev["external_comments"].sum()
t1_rows.append({
    "Cohort": "AIDev v5 (Agentic)",
    "Total Human Comments": f"{aidev_tot_c:,}",
    "Author Self-Comments": f"{aidev_auth_c:,}",
    "Author Share (%)": f"{(aidev_auth_c / aidev_tot_c * 100):.2f}%",
    "External Comments": f"{aidev_ext_c:,}",
    "External Share (%)": f"{(aidev_ext_c / aidev_tot_c * 100):.2f}%"
})

for cohort_name in ["MOSAIC-3M (Agentic Only)", "MOSAIC-3M (Human Baseline)"]:
    sub = df_eval_mosaic[df_eval_mosaic["cohort"] == cohort_name]
    tot_c = sub["total_comments"].sum()
    auth_c = sub["author_comments"].sum()
    ext_c = sub["external_comments"].sum()
    t1_rows.append({
        "Cohort": cohort_name,
        "Total Human Comments": f"{tot_c:,}",
        "Author Self-Comments": f"{auth_c:,}",
        "Author Share (%)": f"{(auth_c / tot_c * 100):.2f}%" if tot_c > 0 else "0.00%",
        "External Comments": f"{ext_c:,}",
        "External Share (%)": f"{(ext_c / tot_c * 100):.2f}%" if tot_c > 0 else "0.00%"
    })

df_t1 = pd.DataFrame(t1_rows)
print(df_t1.to_string(index=False))

# Breakdown in AIDev v5: General Comments vs Inline Review Comments
print("\n--- AIDev v5 Comment Type Authorship Breakdown ---")
gen_tot = df_eval_aidev["author_gen_comments"].sum() + df_eval_aidev["ext_gen_comments"].sum()
gen_auth = df_eval_aidev["author_gen_comments"].sum()
inline_tot = df_eval_aidev["author_inline_comments"].sum() + df_eval_aidev["ext_inline_comments"].sum()
inline_auth = df_eval_aidev["author_inline_comments"].sum()

df_breakdown = pd.DataFrame([
    {
        "Comment Type": "General PR Discussion Comments",
        "Total Comments": f"{gen_tot:,}",
        "Author Self-Comments": f"{gen_auth:,}",
        "Author Share (%)": f"{(gen_auth / gen_tot * 100):.2f}%",
        "External Comments": f"{gen_tot - gen_auth:,}",
        "External Share (%)": f"{((gen_tot - gen_auth) / gen_tot * 100):.2f}%"
    },
    {
        "Comment Type": "Inline Review Comments",
        "Total Comments": f"{inline_tot:,}",
        "Author Self-Comments": f"{inline_auth:,}",
        "Author Share (%)": f"{(inline_auth / inline_tot * 100):.2f}%",
        "External Comments": f"{inline_tot - inline_auth:,}",
        "External Share (%)": f"{((inline_tot - inline_auth) / inline_tot * 100):.2f}%"
    }
])
print(df_breakdown.to_string(index=False))

# --------------------------------------------------------------------------
# 5. Table 2: Silent Integration Reclassification (Before vs. After Purge)
# --------------------------------------------------------------------------
print("\n--- Table 2: Silent Integration Reclassification (Before vs. After Purge) ---")
t2_rows = []

tot_pr = len(df_eval_aidev)
cur_sil = df_eval_aidev["current_silent"].sum()
san_sil = df_eval_aidev["sanitized_silent"].sum()
flipped = df_eval_aidev["at_risk_flip"].sum()
t2_rows.append({
    "Cohort": "AIDev v5 (Agentic)",
    "Total PRs": f"{tot_pr:,}",
    "Current Silent PRs (Count & %)": f"{cur_sil:,} ({(cur_sil / tot_pr * 100):.2f}%)",
    "Sanitized Silent PRs (Count & %)": f"{san_sil:,} ({(san_sil / tot_pr * 100):.2f}%)",
    "Net Flipped PRs (Count & %)": f"{flipped:,} ({(flipped / tot_pr * 100):.2f}%)"
})

for cohort_name in ["MOSAIC-3M (Agentic Only)", "MOSAIC-3M (Human Baseline)"]:
    sub = df_eval_mosaic[df_eval_mosaic["cohort"] == cohort_name]
    tot_pr = len(sub)
    cur_sil = sub["current_silent"].sum()
    san_sil = sub["sanitized_silent"].sum()
    flipped = sub["at_risk_flip"].sum()
    t2_rows.append({
        "Cohort": cohort_name,
        "Total PRs": f"{tot_pr:,}",
        "Current Silent PRs (Count & %)": f"{cur_sil:,} ({(cur_sil / tot_pr * 100):.2f}%)",
        "Sanitized Silent PRs (Count & %)": f"{san_sil:,} ({(san_sil / tot_pr * 100):.2f}%)",
        "Net Flipped PRs (Count & %)": f"{flipped:,} ({(flipped / tot_pr * 100):.2f}%)"
    })

df_t2 = pd.DataFrame(t2_rows)
print(df_t2.to_string(index=False))

# --------------------------------------------------------------------------
# 6. Table 3: Treatment vs. Control Flip Sensitivity
# --------------------------------------------------------------------------
print("\n--- Table 3: Treatment vs. Control Flip Sensitivity ---")
t3_rows = []

cohort_dfs = [
    ("AIDev v5 (Agentic)", df_eval_aidev),
    ("MOSAIC-3M (Agentic Only)", df_eval_mosaic[df_eval_mosaic["cohort"] == "MOSAIC-3M (Agentic Only)"]),
    ("MOSAIC-3M (Human Baseline)", df_eval_mosaic[df_eval_mosaic["cohort"] == "MOSAIC-3M (Human Baseline)"]),
]

for cohort_name, cdf in cohort_dfs:
    for scope_name, label in [("App_and_Infra", "App_and_Infra (Treatment)"), ("App_Only", "App_Only (Control)")]:
        sub = cdf[cdf["scope"] == scope_name]
        tot = len(sub)
        cur_sil = sub["current_silent"].sum()
        san_sil = sub["sanitized_silent"].sum()
        flipped = sub["at_risk_flip"].sum()
        cur_rev = sub["current_reviewed"].sum()
        
        t3_rows.append({
            "Cohort": cohort_name,
            "Scope": label,
            "Total PRs": f"{tot:,}",
            "Current Silent (%)": f"{(cur_sil / tot * 100):.2f}%" if tot > 0 else "N/A",
            "Sanitized Silent (%)": f"{(san_sil / tot * 100):.2f}%" if tot > 0 else "N/A",
            "Flipped PRs": f"{flipped:,}",
            "Flip Rate (% of Cohort)": f"{(flipped / tot * 100):.2f}%" if tot > 0 else "N/A",
            "Flip Rate (% of Reviewed)": f"{(flipped / cur_rev * 100):.2f}%" if cur_rev > 0 else "N/A"
        })

df_t3 = pd.DataFrame(t3_rows)
print(df_t3.to_string(index=False))

# --------------------------------------------------------------------------
# 7. Table 4: Governance Association Audit (MOSAIC-3M Only)
# --------------------------------------------------------------------------
print("\n--- Table 4: Governance Association Audit (MOSAIC-3M Only) ---")
assoc_order = ["OWNER", "MEMBER", "COLLABORATOR", "CONTRIBUTOR", "NONE"]
t4_rows = []
for cohort_name in ["MOSAIC-3M (Agentic Only)", "MOSAIC-3M (Human Baseline)"]:
    cohort_sub = df_eval_mosaic[df_eval_mosaic["cohort"] == cohort_name]
    for assoc in assoc_order:
        sub = cohort_sub[cohort_sub["author_association"] == assoc]
        tot_pr = len(sub)
        if tot_pr == 0:
            continue
        tot_c = sub["total_comments"].sum()
        auth_c = sub["author_comments"].sum()
        cur_sil = sub["current_silent"].sum()
        san_sil = sub["sanitized_silent"].sum()
        flipped = sub["at_risk_flip"].sum()
        
        t4_rows.append({
            "Cohort": cohort_name,
            "Author Association": assoc,
            "PR Count": f"{tot_pr:,}",
            "Total Comments": f"{tot_c:,}",
            "Author Self-Comments": f"{auth_c:,}",
            "Self-Comment Share (%)": f"{(auth_c / tot_c * 100):.2f}%" if tot_c > 0 else "0.00%",
            "Current Silent Rate (%)": f"{(cur_sil / tot_pr * 100):.2f}%",
            "Sanitized Silent Rate (%)": f"{(san_sil / tot_pr * 100):.2f}%",
            "Net Flipped PRs (Count & %)": f"{flipped:,} ({(flipped / tot_pr * 100):.2f}%)"
        })

df_t4 = pd.DataFrame(t4_rows)
print(df_t4.to_string(index=False))


=== 2.2 Diagnostic: PR Author Self-Comment Audit & Reclassification Risk ===


--- Sanity Check: Formal Review Events where Reviewer == PR Author ---
AIDev v5 (Agentic):          717 / 9,262 reviews (7.74%)
MOSAIC-3M (Agentic Only)     158 / 2,484 reviews (6.36%)
MOSAIC-3M (Human Baseline)   1,083 / 3,769 reviews (28.73%)

--- Table 1: Comment Volume & Authorship Audit ---
                    Cohort Total Human Comments Author Self-Comments Author Share (%) External Comments External Share (%)
        AIDev v5 (Agentic)               11,467                1,321           11.52%            10,146             88.48%
  MOSAIC-3M (Agentic Only)                1,392                  299           21.48%             1,093             78.52%
MOSAIC-3M (Human Baseline)                1,788                  709           39.65%             1,079             60.35%

--- AIDev v5 Comment Type Authorship Breakdown ---
                  Comment Type Total Comments Author Self-Comments Author Share (%) External Comments External Share (%)
General PR Discussion Comments        

                    Cohort Author Association PR Count Total Comments Author Self-Comments Self-Comment Share (%) Current Silent Rate (%) Sanitized Silent Rate (%) Net Flipped PRs (Count & %)
  MOSAIC-3M (Agentic Only)              OWNER      261             69                   59                 85.51%                  86.97%                    95.40%                  22 (8.43%)
  MOSAIC-3M (Agentic Only)             MEMBER      201            158                  103                 65.19%                  50.75%                    59.20%                  17 (8.46%)
  MOSAIC-3M (Agentic Only)       COLLABORATOR      138             78                   41                 52.56%                  59.42%                    67.39%                  11 (7.97%)
  MOSAIC-3M (Agentic Only)        CONTRIBUTOR    1,313          1,065                   96                  9.01%                  35.64%                    36.56%                  12 (0.91%)
  MOSAIC-3M (Agentic Only)              

### 2.3 Continuous Comment Volume Shift Analysis (Raw vs. Sanitized)

To quantify the magnitude of comment volume shifts between Raw (total discussion) and Sanitized (external maintainers only) metrics, we evaluate the continuous comment distribution across Treatment and Control cohorts.

This audit establishes the empirical shift in mean and median comment volume, demonstrating the necessity of reporting dual bounds (Raw vs. Sanitized) in our causal evaluations.

In [7]:
print("\n=== 2.3 Diagnostic: Continuous Comment Volume Shift (Raw vs. Sanitized) ===")

# --------------------------------------------------------------------------
# 1. Non-Destructive Copies of Upstream Evaluation DataFrames
# --------------------------------------------------------------------------
df_shift_aidev = df_eval_aidev.copy()
df_shift_mosaic = df_eval_mosaic.copy()

# Add cohort column to AIDev v5 (df_shift_mosaic already has cohort)
df_shift_aidev["cohort"] = "AIDev v5 (Agentic)"

# --------------------------------------------------------------------------
# 2. Concatenation & Population Filtering
# --------------------------------------------------------------------------
cols_to_use = ["cohort", "scope", "total_comments", "external_comments"]
df_shift_eval = pd.concat([df_shift_aidev[cols_to_use], df_shift_mosaic[cols_to_use]], ignore_index=True)

# Filter 1 (Target Population): Isolate PRs with at least one human comment (total_comments > 0)
# Filter 2 (Treatment vs Control): Restrict to 'App_and_Infra' or 'App_Only'
df_shift_filtered = df_shift_eval[
    (df_shift_eval["total_comments"] > 0) & 
    (df_shift_eval["scope"].isin(["App_and_Infra", "App_Only"]))
].copy()

# --------------------------------------------------------------------------
# 3. Grouped Distributional Summary Statistics (Raw vs. Sanitized)
# --------------------------------------------------------------------------
def p75(x):
    return x.quantile(0.75)

agg_df = df_shift_filtered.groupby(["cohort", "scope"]).agg(
    N=("total_comments", "count"),
    raw_mean=("total_comments", "mean"),
    san_mean=("external_comments", "mean"),
    raw_median=("total_comments", "median"),
    san_median=("external_comments", "median"),
    raw_p75=("total_comments", p75),
    san_p75=("external_comments", p75),
    raw_max=("total_comments", "max"),
    san_max=("external_comments", "max"),
).reset_index()

# Calculate Mean Volume Drop (%): (Raw Mean - Sanitized Mean) / Raw Mean * 100
agg_df["mean_drop_pct"] = (agg_df["raw_mean"] - agg_df["san_mean"]) / agg_df["raw_mean"] * 100

# --------------------------------------------------------------------------
# 4. Output Formatting & Clean Contrast
# --------------------------------------------------------------------------
df_volume_shift = pd.DataFrame({
    "Cohort": agg_df["cohort"],
    "Scope": agg_df["scope"],
    "Count (N)": agg_df["N"],
    "Raw Mean": agg_df["raw_mean"].round(2),
    "Sanitized Mean": agg_df["san_mean"].round(2),
    "Mean Volume Drop (%)": agg_df["mean_drop_pct"].round(2),
    "Raw Median": agg_df["raw_median"].round(2),
    "Sanitized Median": agg_df["san_median"].round(2),
    "Raw P75": agg_df["raw_p75"].round(2),
    "Sanitized P75": agg_df["san_p75"].round(2),
    "Raw Max": agg_df["raw_max"],
    "Sanitized Max": agg_df["san_max"],
})

# Sort so that App_and_Infra and App_Only are paired logically for each cohort
cohort_order = {"AIDev v5 (Agentic)": 0, "MOSAIC-3M (Agentic Only)": 1, "MOSAIC-3M (Human Baseline)": 2}
scope_order = {"App_and_Infra": 0, "App_Only": 1}
df_volume_shift["cohort_rank"] = df_volume_shift["Cohort"].map(cohort_order)
df_volume_shift["scope_rank"] = df_volume_shift["Scope"].map(scope_order)
df_volume_shift = df_volume_shift.sort_values(["cohort_rank", "scope_rank"]).drop(columns=["cohort_rank", "scope_rank"])

print(df_volume_shift.to_string(index=False))



=== 2.3 Diagnostic: Continuous Comment Volume Shift (Raw vs. Sanitized) ===
                    Cohort         Scope  Count (N)  Raw Mean  Sanitized Mean  Mean Volume Drop (%)  Raw Median  Sanitized Median  Raw P75  Sanitized P75  Raw Max  Sanitized Max
        AIDev v5 (Agentic) App_and_Infra        531      6.79            6.26                  7.79         5.0               4.0     9.00           8.00       45             45
        AIDev v5 (Agentic)      App_Only       1822      4.32            3.74                 13.23         2.0               2.0     5.00           4.00       77             57
  MOSAIC-3M (Agentic Only) App_and_Infra         40      3.45            2.58                 25.36         3.0               1.5     4.00           4.00       33             20
  MOSAIC-3M (Agentic Only)      App_Only        530      2.37            1.87                 21.05         1.5               1.0     3.00           2.00       23             15
MOSAIC-3M (Human Baseline) App_an

### 2.4 The Baseline "Silent Integration" Phenomenon

Before evaluating causal integration costs, we diagnose the prevalence of "Silent Integrations" — pull requests that are merged with exactly zero human review comments and zero human review iterations (`human_review_comments == 0 & human_review_iterations == 0`).

In empirical software engineering, telemetry allows us to distinguish between:
1. **Silent Integrations:** Pull requests that merge without any documented review discussion or iterative review pushback.
2. **Explicit Review Interventions:** Pull requests that trigger active human review comments or multi-iteration review cycles.

We define a 'Silent Integration' strictly as having `human_review_comments == 0` AND `human_review_iterations == 0`. We evaluate this across the un-matched, cleaned cohorts to establish baseline human intervention behaviors.

In [8]:
print("=== 2.4 The Baseline Silent Integration Phenomenon ===")

def analyze_silent_integrations(df_pr, human_metrics, dataset_name):
    df_eval = df_pr.copy()
    
    # 1. Left join human metrics (bringing in both comments and review iterations)
    df_eval = df_eval[['id', 'scope']].merge(
        human_metrics[['pr_id', 'human_review_comments', 'human_review_iterations']], 
        left_on='id', right_on='pr_id', how='left'
    )
    df_eval['human_review_comments'] = df_eval['human_review_comments'].fillna(0)
    df_eval['human_review_iterations'] = df_eval['human_review_iterations'].fillna(0)
    
    total_prs = len(df_eval)
    if total_prs == 0:
        print(f"No PRs found for {dataset_name}.")
        return
        
    # 2. Define silent integration mask (0 comments AND 0 review iterations)
    silent_integration_mask = (df_eval['human_review_comments'] == 0) & (df_eval['human_review_iterations'] == 0)
    silent_integration_prs = silent_integration_mask.sum()
    
    print(f"\n[{dataset_name}]")
    print(f"Total Evaluated PRs: {total_prs:,}")
    print(f"Silent Integrations (0 Comments & 0 Iterations): {silent_integration_prs:,} ({(silent_integration_prs/total_prs)*100:.1f}%)")
    
    # 3. Breakdown by Scope
    if silent_integration_prs > 0:
        print("\nSilent Integrations by Scope:")
        scope_breakdown = df_eval[silent_integration_mask]['scope'].value_counts().reset_index()
        scope_breakdown.columns = ['Scope', 'Silent Integration PRs']
        
        # Calculate share of that specific scope
        scope_totals = df_eval['scope'].value_counts()
        scope_breakdown['Total PRs in Scope'] = scope_breakdown['Scope'].map(scope_totals)
        scope_breakdown['Share of Scope (%)'] = (scope_breakdown['Silent Integration PRs'] / scope_breakdown['Total PRs in Scope'] * 100).round(2)
        
        print(scope_breakdown.to_string(index=False))

# Run analysis on the pre-matched filtered cohorts
analyze_silent_integrations(df_aidev_pr_filtered, aidev_human_metrics, "AIDev v5")
analyze_silent_integrations(df_mosaic_pr_filtered, mosaic_human_metrics, "MOSAIC-3M (All Agents + Human)")

=== 2.4 The Baseline Silent Integration Phenomenon ===



[AIDev v5]
Total Evaluated PRs: 14,747
Silent Integrations (0 Comments & 0 Iterations): 11,323 (76.8%)

Silent Integrations by Scope:


        Scope  Silent Integration PRs  Total PRs in Scope  Share of Scope (%)
     App_Only                   10996               13768               79.87
App_and_Infra                     327                 979               33.40

[MOSAIC-3M (All Agents + Human)]
Total Evaluated PRs: 3,648
Silent Integrations (0 Comments & 0 Iterations): 1,241 (34.0%)

Silent Integrations by Scope:
        Scope  Silent Integration PRs  Total PRs in Scope  Share of Scope (%)
     App_Only                    1162                3435               33.83
App_and_Infra                      79                 213               37.09


### 2.5 Merge Latency Distribution of Silent vs. Explicitly Reviewed PRs

To test whether our initial 6-minute cutoff ($0 \le \text{latency} < 0.1$ hours) artificially truncated auto-merged PRs delayed by CI/CD execution queues, we evaluate the empirical merge latency distribution across:
1. **Silent Integrations**: PRs with exactly zero human comments AND zero human review iterations (`human_review_comments == 0 & human_review_iterations == 0`).
2. **Explicitly Reviewed PRs**: PRs with at least one human review comment or review iteration.

We compare sample sizes, median latency, mean latency, 75th percentile ($P_{75}$), and 90th percentile ($P_{90}$) across both AIDev v5 and MOSAIC-3M cohorts.

In [9]:
print("=== 2.5 Latency Distribution of Silent vs. Explicitly Reviewed PRs ===")

def diagnose_silent_integration_latency(df_pr, human_metrics, dataset_name):
    # 1. Merge df_pr[['id', 'merged_at', 'created_at']] with human metrics
    df_eval = df_pr[['id', 'merged_at', 'created_at']].merge(
        human_metrics[['pr_id', 'human_review_comments', 'human_review_iterations']],
        left_on='id',
        right_on='pr_id',
        how='left'
    )
    
    # 2. Fill NaNs in human metrics with 0
    df_eval['human_review_comments'] = df_eval['human_review_comments'].fillna(0)
    df_eval['human_review_iterations'] = df_eval['human_review_iterations'].fillna(0)
    
    # 3. Convert merged_at and created_at to datetime (UTC)
    merged_dt = pd.to_datetime(df_eval['merged_at'], utc=True, errors='coerce')
    created_dt = pd.to_datetime(df_eval['created_at'], utc=True, errors='coerce')
    
    # 4. Calculate merge_latency in hours
    df_eval['merge_latency'] = (merged_dt - created_dt).dt.total_seconds() / 3600.0
    
    # 5. Filter clock-skew anomalies (keep only merge_latency >= 0)
    df_eval = df_eval[df_eval['merge_latency'] >= 0].copy()
    
    # 6. Boolean flag for silent integration PRs (0 comments AND 0 review iterations)
    df_eval['is_silent_integration'] = (
        (df_eval['human_review_comments'] == 0) & 
        (df_eval['human_review_iterations'] == 0)
    )
    
    # 7. Group by is_silent_integration and aggregate latency metrics
    summary_table = df_eval.groupby('is_silent_integration').agg(
        count=('merge_latency', 'count'),
        median=('merge_latency', 'median'),
        mean=('merge_latency', 'mean'),
        p75=('merge_latency', lambda x: x.quantile(0.75)),
        p90=('merge_latency', lambda x: x.quantile(0.90))
    ).reset_index()
    
    # 8. Print dataset name and display summary table
    print(f"\n--- {dataset_name}: Merge Latency Distribution by Silent Integration ---")
    display(summary_table.round(2))
    return summary_table

# Run helper function for both datasets
diagnose_silent_integration_latency(df_aidev_pr_filtered, aidev_human_metrics, "AIDev v5")
diagnose_silent_integration_latency(df_mosaic_pr_filtered, mosaic_human_metrics, "MOSAIC-3M")

=== 2.5 Latency Distribution of Silent vs. Explicitly Reviewed PRs ===



--- AIDev v5: Merge Latency Distribution by Silent Integration ---


,is_silent_integration,count,median,mean,p75,p90
0,False,3424,17.44,92.92,79.94,241.37
1,True,11323,0.02,6.63,0.20,1.93



--- MOSAIC-3M: Merge Latency Distribution by Silent Integration ---


,is_silent_integration,count,median,mean,p75,p90
0,False,2407,15.96,84.56,72.86,215.56
1,True,1241,0.37,18.74,3.73,23.22


,is_silent_integration,count,median,mean,p75,p90
0,False,2407,15.961389,84.558640,72.861806,215.557333
1,True,1241,0.365278,18.737139,3.725833,23.223889


### 2.6 Architectural and Agent Profiling of Silent Integrations

We profile the distribution of silent integration PRs across architectural archetypes and individual AI agent sources to examine whether specific agents or project architectures systematically experience higher rates of silent integration.

In [10]:
print("=== 2.6 Profiling Silent Integration PRs (Volume & Agents) ===")

def profile_silent_integration_prs(df_pr, human_metrics, dataset_name, agent_col):
    df_eval = df_pr.copy()
    
    # 1. Left join human metrics (bringing in both comments and review iterations)
    df_eval = df_eval.merge(
        human_metrics[['pr_id', 'human_review_comments', 'human_review_iterations']], 
        left_on='id', right_on='pr_id', how='left'
    )
    df_eval['human_review_comments'] = df_eval['human_review_comments'].fillna(0)
    df_eval['human_review_iterations'] = df_eval['human_review_iterations'].fillna(0)
    
    # 2. Define silent integration mask (0 comments AND 0 review iterations)
    silent_integration_mask = (df_eval['human_review_comments'] == 0) & (df_eval['human_review_iterations'] == 0)
    df_silent = df_eval[silent_integration_mask].copy()
    
    if len(df_silent) == 0:
        return
        
    print(f"\n[{dataset_name} - Silent Integration PR Profile]")
    
    # 1. Volume Profiling
    print("\n1. Code Volume (Median):")
    print(f"   - Total Files Modified: {df_silent['total_files_recorded'].median():.1f}")
    print(f"   - Application Line Churn: {df_silent['app_line_churn'].median():.1f} lines")
    
    # 2. Agent Profiling
    print("\n2. Agent Distribution (Who is making these?):")
    agent_dist = df_silent[agent_col].value_counts().reset_index()
    agent_dist.columns = ['Agent', 'Silent Integration PRs']
    agent_dist['Share (%)'] = (agent_dist['Silent Integration PRs'] / len(df_silent) * 100).round(1)
    print(agent_dist.to_string(index=False))

# Run profiling
profile_silent_integration_prs(df_aidev_pr_filtered, aidev_human_metrics, "AIDev v5", "agent")
profile_silent_integration_prs(df_mosaic_pr_filtered, mosaic_human_metrics, "MOSAIC-3M", "agent_source")

=== 2.6 Profiling Silent Integration PRs (Volume & Agents) ===

[AIDev v5 - Silent Integration PR Profile]

1. Code Volume (Median):
   - Total Files Modified: 4.0
   - Application Line Churn: 46.0 lines

2. Agent Distribution (Who is making these?):
       Agent  Silent Integration PRs  Share (%)
OpenAI_Codex                   10211       90.2
      Cursor                     588        5.2
     Copilot                     207        1.8
       Devin                     161        1.4
 Claude_Code                     141        1.2
Google_Jules                      15        0.1

[MOSAIC-3M - Silent Integration PR Profile]

1. Code Volume (Median):
   - Total Files Modified: 4.0
   - Application Line Churn: 51.0 lines

2. Agent Distribution (Who is making these?):
  Agent  Silent Integration PRs  Share (%)
 Claude                     390       31.4
  Human                     362       29.2
  Devin                     243       19.6
  Codex                     196       15.8
Copilot  

## Section 3: Matched Cohort Linking and Baseline 1 Evaluation
### 3.1 Causal Matched Cohort Ingestion & Metric Merging

To evaluate the causal effect of infrastructure co-modifications on human review friction, we link our PR-level human review metrics onto the exact-matched cohorts generated in RQ2 (`df_aidev_matched`, `df_mosaic_matched`).

We left-merge the engineered metrics (`human_review_iterations`, `human_review_comments`, `external_review_iterations`, `external_comments`) onto the matched treatment and control units to prepare for causal ATT evaluation.

In [11]:
# 4. Generate Matched Cohorts and Left-Merge Human Metrics
# --------------------------------------------------------------------------
df_aidev_matched, df_mosaic_matched = generate_matched_cohorts(df_aidev_pr_filtered, df_mosaic_pr_filtered)

# Left-merge onto df_aidev_matched
df_aidev_matched = df_aidev_matched.merge(aidev_human_metrics, left_on="id", right_on="pr_id", how="left")
df_aidev_matched["human_review_iterations"] = df_aidev_matched["human_review_iterations"].fillna(0).astype(int)
df_aidev_matched["human_review_comments"] = df_aidev_matched["human_review_comments"].fillna(0).astype(int)
if "pr_id_y" in df_aidev_matched.columns: df_aidev_matched.drop(columns=["pr_id_y"], inplace=True)

df_aidev_matched = df_aidev_matched.merge(
    df_eval_aidev[["id", "external_comments", "external_review_iterations"]],
    on="id",
    how="left"
)
df_aidev_matched["external_comments"] = df_aidev_matched["external_comments"].fillna(0).astype(int)
df_aidev_matched["external_review_iterations"] = df_aidev_matched["external_review_iterations"].fillna(0).astype(int)

# Define boolean silent integration flags (Raw vs. Sanitized)
df_aidev_matched["is_silent"] = (df_aidev_matched["human_review_comments"] == 0) & (df_aidev_matched["human_review_iterations"] == 0)
df_aidev_matched["sanitized_silent"] = (df_aidev_matched["external_comments"] == 0) & (df_aidev_matched["external_review_iterations"] == 0)

# Left-merge onto df_mosaic_matched
df_mosaic_matched = df_mosaic_matched.merge(mosaic_human_metrics, left_on="id", right_on="pr_id", how="left")
df_mosaic_matched["human_review_iterations"] = df_mosaic_matched["human_review_iterations"].fillna(0).astype(int)
df_mosaic_matched["human_review_comments"] = df_mosaic_matched["human_review_comments"].fillna(0).astype(int)
if "pr_id_y" in df_mosaic_matched.columns: df_mosaic_matched.drop(columns=["pr_id_y"], inplace=True)

df_mosaic_matched = df_mosaic_matched.merge(
    df_eval_mosaic[["id", "external_comments", "external_review_iterations"]],
    on="id",
    how="left"
)
df_mosaic_matched["external_comments"] = df_mosaic_matched["external_comments"].fillna(0).astype(int)
df_mosaic_matched["external_review_iterations"] = df_mosaic_matched["external_review_iterations"].fillna(0).astype(int)

df_mosaic_matched["is_silent"] = (df_mosaic_matched["human_review_comments"] == 0) & (df_mosaic_matched["human_review_iterations"] == 0)
df_mosaic_matched["sanitized_silent"] = (df_mosaic_matched["external_comments"] == 0) & (df_mosaic_matched["external_review_iterations"] == 0)

# --------------------------------------------------------------------------
# 5. Descriptive Statistics of Human Metrics Across Matched Cohorts
# --------------------------------------------------------------------------
metric_cols = ["human_review_iterations", "human_review_comments", "external_review_iterations", "external_comments"]

print("\n--- AIDev v5 (Matched): Human Review Metrics Distribution ---")
print(df_aidev_matched[metric_cols].describe().round(2).to_string())

print("\n--- MOSAIC-3M (Matched): Human Review Metrics Distribution ---")
print(df_mosaic_matched[metric_cols].describe().round(2).to_string())



Executing Greedy 1:1 Matching: AIDev v5
Treatment: scope == 'App_and_Infra' | Control: 'App_Only'
Strata: ['agent', 'architectural_archetype', 'has_tests', 'ecosystem'] | Distance: ['z_log_app_count']


Successfully Matched: 687 pairs

Executing Greedy 1:1 Matching: MOSAIC-3M (Agentic Only)
Treatment: scope == 'App_and_Infra' | Control: 'App_Only'
Strata: ['agent_source', 'architectural_archetype', 'has_tests', 'ecosystem'] | Distance: ['z_log_app_count']


Successfully Matched: 80 pairs

--- AIDev v5 (Matched): Human Review Metrics Distribution ---
       human_review_iterations  human_review_comments  external_review_iterations  external_comments
count                  1374.00                1374.00                     1374.00            1374.00
mean                      2.05                   2.82                        1.92               2.59
std                       3.27                   5.54                        3.17               5.39
min                       0.00                   0.00                        0.00               0.00
25%                       0.00                   0.00                        0.00               0.00
50%                       1.00                   0.00                        1.00               0.00
75%                       3.00                   3.00                        2.75               3.00
max                      23.00                  52.00                       23.00              52.

### 3.2 Baseline 1: Agentic Review Overhead (Treatment vs. Control ATT)

We evaluate the causal treatment effect of co-changing infrastructure files (`App_and_Infra` Treatment vs. `App_Only` Control) on human intervention overhead for AI agents:
- **Matching Strata**: Exact matching on `(agent, architectural_archetype, has_tests, ecosystem)` minimizing 1D standardized distance on application volume (`z_log_app_count`) with a strict $0.28$ caliper.
- **Average Treatment Effect on the Treated (ATT)**: Mean paired difference across exact-matched pairs.
- **Intervention Rate**: Proportion of PRs requiring at least one human review iteration or comment.

*Methodological Note on Dual-Bound Evaluation: We evaluate review friction under two empirical bounds: Bound 1 (Raw / Upper Bound) captures total human discussion including author self-comments; Bound 2 (Sanitized / Primary Conservative Bound) strictly isolates external maintainer oversight.*

In [12]:
print("=== 3.2 Baseline 1 Evaluation: Agentic Review Overhead (ATT) ===")

def evaluate_baseline1_att(df_matched: pd.DataFrame, dataset_name: str):
    paired_df = df_matched.pivot(
        index="pair_id", 
        columns="match_role", 
        values=["human_review_iterations", "human_review_comments", "external_review_iterations", "external_comments"]
    )
    
    # Bound 1: Raw (Total Friction)
    raw_iterations_diff = paired_df["human_review_iterations"]["Treatment"] - paired_df["human_review_iterations"]["Control"]
    raw_comments_diff = paired_df["human_review_comments"]["Treatment"] - paired_df["human_review_comments"]["Control"]
    
    treat_raw_iterations_rate = (paired_df["human_review_iterations"]["Treatment"] > 0).mean() * 100.0
    ctrl_raw_iterations_rate = (paired_df["human_review_iterations"]["Control"] > 0).mean() * 100.0
    
    treat_raw_comments_rate = (paired_df["human_review_comments"]["Treatment"] > 0).mean() * 100.0
    ctrl_raw_comments_rate = (paired_df["human_review_comments"]["Control"] > 0).mean() * 100.0

    # Bound 2: Sanitized (External Friction)
    san_iterations_diff = paired_df["external_review_iterations"]["Treatment"] - paired_df["external_review_iterations"]["Control"]
    san_comments_diff = paired_df["external_comments"]["Treatment"] - paired_df["external_comments"]["Control"]

    treat_san_iterations_rate = (paired_df["external_review_iterations"]["Treatment"] > 0).mean() * 100.0
    ctrl_san_iterations_rate = (paired_df["external_review_iterations"]["Control"] > 0).mean() * 100.0

    treat_san_comments_rate = (paired_df["external_comments"]["Treatment"] > 0).mean() * 100.0
    ctrl_san_comments_rate = (paired_df["external_comments"]["Control"] > 0).mean() * 100.0
    
    print(f"\n--- Baseline 1 ATT: {dataset_name} ---")
    print("Bound 1: Raw (Total Friction)")
    print("  Metric 1: Human Review Iterations")
    print(f"    - Mean Paired Difference: {raw_iterations_diff.mean():+.2f} iterations")
    print(f"    - Intervention Rate: Treatment {treat_raw_iterations_rate:.1f}% vs Control {ctrl_raw_iterations_rate:.1f}%")
    print("  Metric 2: Human Review Comments")
    print(f"    - Mean Paired Difference: {raw_comments_diff.mean():+.2f} comments")
    print(f"    - Intervention Rate: Treatment {treat_raw_comments_rate:.1f}% vs Control {ctrl_raw_comments_rate:.1f}%")

    print("\nBound 2: Sanitized (External Friction)")
    print("  Metric 1: External Review Iterations")
    print(f"    - Mean Paired Difference: {san_iterations_diff.mean():+.2f} iterations")
    print(f"    - Intervention Rate: Treatment {treat_san_iterations_rate:.1f}% vs Control {ctrl_san_iterations_rate:.1f}%")
    print("  Metric 2: External Comments")
    print(f"    - Mean Paired Difference: {san_comments_diff.mean():+.2f} comments")
    print(f"    - Intervention Rate: Treatment {treat_san_comments_rate:.1f}% vs Control {ctrl_san_comments_rate:.1f}%")

evaluate_baseline1_att(df_aidev_matched, "AIDev v5")
evaluate_baseline1_att(df_mosaic_matched, "MOSAIC-3M (Agentic Only)")


=== 3.2 Baseline 1 Evaluation: Agentic Review Overhead (ATT) ===

--- Baseline 1 ATT: AIDev v5 ---
Bound 1: Raw (Total Friction)
  Metric 1: Human Review Iterations
    - Mean Paired Difference: +0.62 iterations
    - Intervention Rate: Treatment 56.6% vs Control 49.1%
  Metric 2: Human Review Comments
    - Mean Paired Difference: +0.78 comments
    - Intervention Rate: Treatment 50.4% vs Control 42.5%

Bound 2: Sanitized (External Friction)
  Metric 1: External Review Iterations
    - Mean Paired Difference: +0.47 iterations
    - Intervention Rate: Treatment 53.1% vs Control 47.5%
  Metric 2: External Comments
    - Mean Paired Difference: +0.60 comments
    - Intervention Rate: Treatment 44.8% vs Control 38.7%

--- Baseline 1 ATT: MOSAIC-3M (Agentic Only) ---
Bound 1: Raw (Total Friction)
  Metric 1: Human Review Iterations
    - Mean Paired Difference: -0.28 iterations
    - Intervention Rate: Treatment 38.8% vs Control 52.5%
  Metric 2: Human Review Comments
    - Mean Paired Dif

## Section 4: Cross-Sectional Human Baseline Equivalence and Intervention Bifurcation
### 4.1 Baseline 2 Matching Execution: Human App+Infra vs. Human App-Only

We evaluate the baseline integration overhead when human developers introduce infrastructure modifications:
- **Treatment**: Human developer introducing dual-scope modifications (`scope == 'App_and_Infra'`).
- **Control**: Human developer introducing application-only changes (`scope == 'App_Only'`).
- **Matching Strata**: Exact matching on `(architectural_archetype, has_tests, ecosystem)` minimizing 1D standardized distance on application volume (`z_log_app_count`) with a strict caliper of $0.28$.

*Methodological Note on Dual-Bound Evaluation: We evaluate review friction under two empirical bounds: Bound 1 (Raw / Upper Bound) captures total human discussion including author self-comments; Bound 2 (Sanitized / Primary Conservative Bound) strictly isolates external maintainer oversight.*

In [13]:
print("=== 4.1 Baseline 2 Evaluation: Human Infrastructure Review Overhead ===")

# --------------------------------------------------------------------------
# Step 1: Prepare the Human Cohort
# --------------------------------------------------------------------------
# Filter df_mosaic_pr_filtered to isolate only Human PRs
df_mosaic_human = df_mosaic_pr_filtered[df_mosaic_pr_filtered["agent_source"] == "Human"].copy()

# Ensure matching covariates are present
if "z_log_app_count" not in df_mosaic_human.columns:
    df_mosaic_human["log_app_count"] = np.log(df_mosaic_human["app_count"])
    df_mosaic_human["z_log_app_count"] = (
        df_mosaic_human["log_app_count"] - df_mosaic_human["log_app_count"].mean()
    ) / df_mosaic_human["log_app_count"].std()

if "ecosystem" not in df_mosaic_human.columns and "primary_language" in df_mosaic_human.columns:
    df_mosaic_human["ecosystem"] = df_mosaic_human["primary_language"].apply(map_ecosystem)

# --------------------------------------------------------------------------
# Step 2: Execute Matching (1D on z_log_app_count)
# --------------------------------------------------------------------------
df_mosaic_human_matched = perform_greedy_strata_matching(
    df=df_mosaic_human,
    treatment_col="scope",
    treatment_val="App_and_Infra",
    control_val="App_Only",
    strata_cols=["architectural_archetype", "has_tests", "ecosystem"],
    distance_cols=["z_log_app_count"],
    caliper_threshold=0.28,
    dataset_name="MOSAIC-3M (Human App+Infra vs. Human App-Only)"
)

# --------------------------------------------------------------------------
# Step 3: Merge Metrics & Evaluate Human ATT
# --------------------------------------------------------------------------
# Left-merge mosaic_human_metrics onto df_mosaic_human_matched
df_mosaic_human_matched = df_mosaic_human_matched.merge(
    mosaic_human_metrics, left_on="id", right_on="pr_id", how="left"
)
df_mosaic_human_matched["human_review_iterations"] = df_mosaic_human_matched["human_review_iterations"].fillna(0).astype(int)
df_mosaic_human_matched["human_review_comments"] = df_mosaic_human_matched["human_review_comments"].fillna(0).astype(int)
if "pr_id_y" in df_mosaic_human_matched.columns:
    df_mosaic_human_matched.drop(columns=["pr_id_y"], inplace=True)
if "pr_id_x" in df_mosaic_human_matched.columns:
    df_mosaic_human_matched.rename(columns={"pr_id_x": "pr_id"}, inplace=True)

# Merge sanitized metrics onto df_mosaic_human_matched
df_mosaic_human_matched = df_mosaic_human_matched.merge(
    df_eval_mosaic[["id", "external_comments", "external_review_iterations"]],
    on="id",
    how="left"
)
df_mosaic_human_matched["external_comments"] = df_mosaic_human_matched["external_comments"].fillna(0).astype(int)
df_mosaic_human_matched["external_review_iterations"] = df_mosaic_human_matched["external_review_iterations"].fillna(0).astype(int)

# Define boolean silent integration flags
df_mosaic_human_matched["is_silent"] = (df_mosaic_human_matched["human_review_comments"] == 0) & (df_mosaic_human_matched["human_review_iterations"] == 0)
df_mosaic_human_matched["sanitized_silent"] = (df_mosaic_human_matched["external_comments"] == 0) & (df_mosaic_human_matched["external_review_iterations"] == 0)

paired_df_human = df_mosaic_human_matched.pivot(
    index="pair_id", 
    columns="match_role", 
    values=["human_review_iterations", "human_review_comments", "external_review_iterations", "external_comments"]
)

# Bound 1: Raw (Total Friction)
human_raw_iterations_diff = paired_df_human["human_review_iterations"]["Treatment"] - paired_df_human["human_review_iterations"]["Control"]
human_raw_comments_diff = paired_df_human["human_review_comments"]["Treatment"] - paired_df_human["human_review_comments"]["Control"]

human_treat_raw_it_rate = (paired_df_human["human_review_iterations"]["Treatment"] > 0).mean() * 100.0
human_ctrl_raw_it_rate = (paired_df_human["human_review_iterations"]["Control"] > 0).mean() * 100.0

human_treat_raw_comm_rate = (paired_df_human["human_review_comments"]["Treatment"] > 0).mean() * 100.0
human_ctrl_raw_comm_rate = (paired_df_human["human_review_comments"]["Control"] > 0).mean() * 100.0

human_raw_iterations_att = float(human_raw_iterations_diff.mean())
human_raw_comments_att = float(human_raw_comments_diff.mean())

# Bound 2: Sanitized (External Friction)
human_san_iterations_diff = paired_df_human["external_review_iterations"]["Treatment"] - paired_df_human["external_review_iterations"]["Control"]
human_san_comments_diff = paired_df_human["external_comments"]["Treatment"] - paired_df_human["external_comments"]["Control"]

human_treat_san_it_rate = (paired_df_human["external_review_iterations"]["Treatment"] > 0).mean() * 100.0
human_ctrl_san_it_rate = (paired_df_human["external_review_iterations"]["Control"] > 0).mean() * 100.0

human_treat_san_comm_rate = (paired_df_human["external_comments"]["Treatment"] > 0).mean() * 100.0
human_ctrl_san_comm_rate = (paired_df_human["external_comments"]["Control"] > 0).mean() * 100.0

human_san_iterations_att = float(human_san_iterations_diff.mean())
human_san_comments_att = float(human_san_comments_diff.mean())

print(f"\n--- Baseline 2 ATT: MOSAIC-3M (Human App+Infra vs. Human App-Only) ---")
print(f"Sample: {len(paired_df_human):,} Matched Pairs")
print("Bound 1: Raw (Total Friction)")
print("  Metric 1: Human Review Iterations")
print(f"    - Mean Paired Difference (ATT): {human_raw_iterations_att:+.2f} iterations")
print(f"    - Intervention Rate: Treatment {human_treat_raw_it_rate:.1f}% vs Control {human_ctrl_raw_it_rate:.1f}%")
print("  Metric 2: Human Review Comments")
print(f"    - Mean Paired Difference (ATT): {human_raw_comments_att:+.2f} comments")
print(f"    - Intervention Rate: Treatment {human_treat_raw_comm_rate:.1f}% vs Control {human_ctrl_raw_comm_rate:.1f}%")

print("\nBound 2: Sanitized (External Friction)")
print("  Metric 1: External Review Iterations")
print(f"    - Mean Paired Difference (ATT): {human_san_iterations_att:+.2f} iterations")
print(f"    - Intervention Rate: Treatment {human_treat_san_it_rate:.1f}% vs Control {human_ctrl_san_it_rate:.1f}%")
print("  Metric 2: External Comments")
print(f"    - Mean Paired Difference (ATT): {human_san_comments_att:+.2f} comments")
print(f"    - Intervention Rate: Treatment {human_treat_san_comm_rate:.1f}% vs Control {human_ctrl_san_comm_rate:.1f}%")


=== 4.1 Baseline 2 Evaluation: Human Infrastructure Review Overhead ===

Executing Greedy 1:1 Matching: MOSAIC-3M (Human App+Infra vs. Human App-Only)
Treatment: scope == 'App_and_Infra' | Control: 'App_Only'
Strata: ['architectural_archetype', 'has_tests', 'ecosystem'] | Distance: ['z_log_app_count']


Successfully Matched: 79 pairs

--- Baseline 2 ATT: MOSAIC-3M (Human App+Infra vs. Human App-Only) ---
Sample: 79 Matched Pairs
Bound 1: Raw (Total Friction)
  Metric 1: Human Review Iterations
    - Mean Paired Difference (ATT): -2.52 iterations
    - Intervention Rate: Treatment 73.4% vs Control 73.4%
  Metric 2: Human Review Comments
    - Mean Paired Difference (ATT): -0.27 comments
    - Intervention Rate: Treatment 34.2% vs Control 48.1%

Bound 2: Sanitized (External Friction)
  Metric 1: External Review Iterations
    - Mean Paired Difference (ATT): -1.42 iterations
    - Intervention Rate: Treatment 73.4% vs Control 72.2%
  Metric 2: External Comments
    - Mean Paired Difference (ATT): +0.01 comments
    - Intervention Rate: Treatment 26.6% vs Control 38.0%


### 4.2 Cross-Sectional Baseline Equivalence Verification (Median Volume Parity)

To verify that our Cross-Sectional Interaction Effect baselines are methodologically sound and directly comparable, we evaluate the baseline equivalence of the Control cohorts (`App_Only` PRs) across the Agentic and Human tracks. We compare the distribution and summary statistics of absolute application file modifications (`app_count`) between the Agent Control and Human Control groups.

In [14]:
# Extract the Control groups (App-Only PRs) from both matched cohorts
agent_control = df_mosaic_matched[df_mosaic_matched['match_role'] == 'Control']['app_count']
human_control = df_mosaic_human_matched[df_mosaic_human_matched['match_role'] == 'Control']['app_count']

# Create a comparison dataframe
baseline_comparison = pd.DataFrame({
    "Metric": ["Count", "Mean App Files", "Median App Files", "Std Dev", "Min", "Max"],
    "Agent Baseline (App-Only)": [
        agent_control.count(),
        round(agent_control.mean(), 2),
        agent_control.median(),
        round(agent_control.std(), 2),
        agent_control.min(),
        agent_control.max()
    ],
    "Human Baseline (App-Only)": [
        human_control.count(),
        round(human_control.mean(), 2),
        human_control.median(),
        round(human_control.std(), 2),
        human_control.min(),
        human_control.max()
    ]
})

print("=== 4.2 Cross-Sectional Baseline Equivalence Check (Median Volume Balance) ===")
print(baseline_comparison.to_string(index=False))

=== 4.2 Cross-Sectional Baseline Equivalence Check (Median Volume Balance) ===
          Metric  Agent Baseline (App-Only)  Human Baseline (App-Only)
           Count                      80.00                      79.00
  Mean App Files                       7.45                      11.89
Median App Files                       4.00                       4.00
         Std Dev                      10.39                      19.84
             Min                       1.00                       1.00
             Max                      64.00                      88.00


### 4.3 The Intervention Bifurcation: Human vs. Agent Oversight Deltas

> **Methodological Note (Cross-Sectional Interaction Effect & The Intervention Bifurcation)**: Directly matching Agents to Humans on dual-scope PRs failed due to a severe lack of common support in covariate distributions (the Curse of Dimensionality, yielding $N=9$ pairs). Instead, we utilize a Cross-Sectional Interaction Effect approach: we evaluate the explicit review intervention rates of adding infrastructure for Agents (Baseline 1) versus Humans (Baseline 2), measuring the percentage point shifts (**Intervention Deltas**) to isolate the bifurcation in human oversight between human and agentic contributions.

In [15]:
# --------------------------------------------------------------------------
# Step 4: Cross-Sectional Interaction Effect (The Intervention Delta)
# --------------------------------------------------------------------------
# Re-evaluate Agent rates from df_mosaic_matched for comparison
paired_df_agent = df_mosaic_matched.pivot(
    index="pair_id", 
    columns="match_role", 
    values=["human_review_iterations", "human_review_comments", "external_review_iterations", "external_comments"]
)

# Raw Bound
agent_treat_raw_it_rate = (paired_df_agent["human_review_iterations"]["Treatment"] > 0).mean() * 100.0
agent_ctrl_raw_it_rate = (paired_df_agent["human_review_iterations"]["Control"] > 0).mean() * 100.0

agent_treat_raw_comm_rate = (paired_df_agent["human_review_comments"]["Treatment"] > 0).mean() * 100.0
agent_ctrl_raw_comm_rate = (paired_df_agent["human_review_comments"]["Control"] > 0).mean() * 100.0

human_raw_it_delta = human_treat_raw_it_rate - human_ctrl_raw_it_rate
human_raw_comm_delta = human_treat_raw_comm_rate - human_ctrl_raw_comm_rate

agent_raw_it_delta = agent_treat_raw_it_rate - agent_ctrl_raw_it_rate
agent_raw_comm_delta = agent_treat_raw_comm_rate - agent_ctrl_raw_comm_rate

# Sanitized Bound
agent_treat_san_it_rate = (paired_df_agent["external_review_iterations"]["Treatment"] > 0).mean() * 100.0
agent_ctrl_san_it_rate = (paired_df_agent["external_review_iterations"]["Control"] > 0).mean() * 100.0

agent_treat_san_comm_rate = (paired_df_agent["external_comments"]["Treatment"] > 0).mean() * 100.0
agent_ctrl_san_comm_rate = (paired_df_agent["external_comments"]["Control"] > 0).mean() * 100.0

human_san_it_delta = human_treat_san_it_rate - human_ctrl_san_it_rate
human_san_comm_delta = human_treat_san_comm_rate - human_ctrl_san_comm_rate

agent_san_it_delta = agent_treat_san_it_rate - agent_ctrl_san_it_rate
agent_san_comm_delta = agent_treat_san_comm_rate - agent_ctrl_san_comm_rate

print(f"\n==================================================================")
print(f"  CROSS-SECTIONAL INTERACTION: THE INTERVENTION BIFURCATION (MOSAIC-3M)")
print(f"==================================================================")
print("Bound 1: Raw (Total Friction)")
print("  Metric 1: Human Review Iterations (Explicit Intervention Rate)")
print(f"    - Human Baseline (App-Only -> App+Infra) : {human_ctrl_raw_it_rate:.1f}% -> {human_treat_raw_it_rate:.1f}% (Delta: {human_raw_it_delta:+.1f} pp)")
print(f"    - Agent Baseline (App-Only -> App+Infra) : {agent_ctrl_raw_it_rate:.1f}% -> {agent_treat_raw_it_rate:.1f}% (Delta: {agent_raw_it_delta:+.1f} pp)")

print("\n  Metric 2: Human Review Comments (Explicit Intervention Rate)")
print(f"    - Human Baseline (App-Only -> App+Infra) : {human_ctrl_raw_comm_rate:.1f}% -> {human_treat_raw_comm_rate:.1f}% (Delta: {human_raw_comm_delta:+.1f} pp)")
print(f"    - Agent Baseline (App-Only -> App+Infra) : {agent_ctrl_raw_comm_rate:.1f}% -> {agent_treat_raw_comm_rate:.1f}% (Delta: {agent_raw_comm_delta:+.1f} pp)")

print("\nBound 2: Sanitized (External Friction)")
print("  Metric 1: External Review Iterations (Explicit Intervention Rate)")
print(f"    - Human Baseline (App-Only -> App+Infra) : {human_ctrl_san_it_rate:.1f}% -> {human_treat_san_it_rate:.1f}% (Delta: {human_san_it_delta:+.1f} pp)")
print(f"    - Agent Baseline (App-Only -> App+Infra) : {agent_ctrl_san_it_rate:.1f}% -> {agent_treat_san_it_rate:.1f}% (Delta: {agent_san_it_delta:+.1f} pp)")

print("\n  Metric 2: External Comments (Explicit Intervention Rate)")
print(f"    - Human Baseline (App-Only -> App+Infra) : {human_ctrl_san_comm_rate:.1f}% -> {human_treat_san_comm_rate:.1f}% (Delta: {human_san_comm_delta:+.1f} pp)")
print(f"    - Agent Baseline (App-Only -> App+Infra) : {agent_ctrl_san_comm_rate:.1f}% -> {agent_treat_san_comm_rate:.1f}% (Delta: {agent_san_comm_delta:+.1f} pp)")



  CROSS-SECTIONAL INTERACTION: THE INTERVENTION BIFURCATION (MOSAIC-3M)
Bound 1: Raw (Total Friction)
  Metric 1: Human Review Iterations (Explicit Intervention Rate)
    - Human Baseline (App-Only -> App+Infra) : 73.4% -> 73.4% (Delta: +0.0 pp)
    - Agent Baseline (App-Only -> App+Infra) : 52.5% -> 38.8% (Delta: -13.8 pp)

  Metric 2: Human Review Comments (Explicit Intervention Rate)
    - Human Baseline (App-Only -> App+Infra) : 48.1% -> 34.2% (Delta: -13.9 pp)
    - Agent Baseline (App-Only -> App+Infra) : 35.0% -> 30.0% (Delta: -5.0 pp)

Bound 2: Sanitized (External Friction)
  Metric 1: External Review Iterations (Explicit Intervention Rate)
    - Human Baseline (App-Only -> App+Infra) : 72.2% -> 73.4% (Delta: +1.3 pp)
    - Agent Baseline (App-Only -> App+Infra) : 52.5% -> 36.2% (Delta: -16.2 pp)

  Metric 2: External Comments (Explicit Intervention Rate)
    - Human Baseline (App-Only -> App+Infra) : 38.0% -> 26.6% (Delta: -11.4 pp)
    - Agent Baseline (App-Only -> App+Infra

### 4.4 Paired Silent Integration Odds Ratios (Exact McNemar Test)

Additionally, we evaluate the Odds Ratios of pull requests achieving **Silent Integration** (defined deterministically as `human_review_comments == 0` and `human_review_iterations == 0`) across both agentic matched cohorts (`df_aidev_matched`, `df_mosaic_matched`) and the human matched cohort (`df_mosaic_human_matched`) using the Exact McNemar Test (paired binary test) and calculate the Matched (Conditional) Odds Ratio with 95% Confidence Intervals.

In [16]:
# --------------------------------------------------------------------------
# Step 5: "Silent Integration" Odds Ratios (Exact McNemar Test & Matched OR)
# --------------------------------------------------------------------------
from scipy.stats import binomtest

def evaluate_silent_integration_odds_ratio(df_matched: pd.DataFrame, dataset_name: str, metric_col: str, bound_name: str) -> dict:
    df = df_matched.copy()
    
    # Defensive guard before reshaping
    df = df.drop_duplicates(subset=["pair_id", "match_role"])
    
    paired = df.pivot(index="pair_id", columns="match_role", values=metric_col)
    treat_silent = paired["Treatment"]
    ctrl_silent = paired["Control"]
    
    # Paired 2x2 Contingency Table states:
    a = int(((treat_silent == True) & (ctrl_silent == True)).sum())   # Both Silent (Concordant)
    b = int(((treat_silent == True) & (ctrl_silent == False)).sum())  # Treatment Only (Discordant)
    c = int(((treat_silent == False) & (ctrl_silent == True)).sum())  # Control Only (Discordant)
    d = int(((treat_silent == False) & (ctrl_silent == False)).sum()) # Both Reviewed (Concordant)
    
    n_pairs = len(paired)
    treat_rate = (treat_silent.sum() / n_pairs * 100.0) if n_pairs > 0 else 0.0
    ctrl_rate = (ctrl_silent.sum() / n_pairs * 100.0) if n_pairs > 0 else 0.0
    
    # Exact McNemar Test (Two-sided binomial test on discordant pairs)
    disc_total = b + c
    if disc_total == 0:
        p_val = 1.0
    else:
        p_val = float(binomtest(k=b, n=disc_total, p=0.5, alternative='two-sided').pvalue)
        
    # Matched (Conditional) Odds Ratio (OR_matched) & 95% CI
    if b == 0 and c == 0:
        odds_ratio = 1.00
        ci_lower = 1.00
        ci_upper = 1.00
    elif b == 0 or c == 0:
        # Haldane-Anscombe correction
        b_adj = b + 0.5
        c_adj = c + 0.5
        odds_ratio = b_adj / c_adj
        se = np.sqrt(1.0 / b_adj + 1.0 / c_adj)
        ci_lower = float(np.exp(np.log(odds_ratio) - 1.96 * se))
        ci_upper = float(np.exp(np.log(odds_ratio) + 1.96 * se))
    else:
        odds_ratio = float(b) / float(c)
        se = np.sqrt(1.0 / b + 1.0 / c)
        ci_lower = float(np.exp(np.log(odds_ratio) - 1.96 * se))
        ci_upper = float(np.exp(np.log(odds_ratio) + 1.96 * se))
        
    print(f"\n[{dataset_name} - {bound_name}]")
    print(f"  Paired 2x2 Table: [Both Silent (a): {a}, Treat Only (b): {b}, Ctrl Only (c): {c}, Both Reviewed (d): {d}]")
    print(f"  - Matched Odds Ratio (OR): {odds_ratio:.4f} (95% CI: [{ci_lower:.4f}, {ci_upper:.4f}])")
    print(f"  - Exact McNemar p-value   : {p_val:.4e} {'(Statistically Significant)' if p_val < 0.05 else '(Not Significant)'}")
    
    return {
        "Cohort": dataset_name,
        "Metric Bound": bound_name,
        "Matched Pairs (N)": n_pairs,
        "Both Silent (a)": a,
        "Treat Only (b)": b,
        "Ctrl Only (c)": c,
        "Both Reviewed (d)": d,
        "Treat Silent Rate (%)": f"{treat_rate:.1f}%",
        "Ctrl Silent Rate (%)": f"{ctrl_rate:.1f}%",
        "Matched OR": f"{odds_ratio:.2f}",
        "95% CI": f"[{ci_lower:.2f}, {ci_upper:.2f}]",
        "p-value (Exact McNemar)": f"{p_val:.4e}"
    }

cohorts = [
    (df_aidev_matched, "AIDev v5 (Agentic)"),
    (df_mosaic_matched, "MOSAIC-3M (Agentic Only)"),
    (df_mosaic_human_matched, "MOSAIC-3M (Human Baseline)")
]

bounds = [
    ("is_silent", "Raw (Total Friction)"),
    ("sanitized_silent", "Sanitized (External Friction)")
]

print("\n" + "=" * 66)
print("  SILENT INTEGRATION MATCHED ODDS RATIOS (EXACT MCNEMAR TEST)")
print("=" * 66)

silent_or_results = []
for df, name in cohorts:
    for metric_col, bound_name in bounds:
        res = evaluate_silent_integration_odds_ratio(df, name, metric_col, bound_name)
        silent_or_results.append(res)

# Summary Table
df_silent_summary = pd.DataFrame(silent_or_results)
print("\n[Summary Table: Silent Integration Matched Odds Ratios (Exact McNemar)]")
print(df_silent_summary.to_string(index=False))



  SILENT INTEGRATION MATCHED ODDS RATIOS (EXACT MCNEMAR TEST)

[AIDev v5 (Agentic) - Raw (Total Friction)]
  Paired 2x2 Table: [Both Silent (a): 202, Treat Only (b): 59, Ctrl Only (c): 97, Both Reviewed (d): 329]
  - Matched Odds Ratio (OR): 0.6082 (95% CI: [0.4401, 0.8407])
  - Exact McNemar p-value   : 2.9306e-03 (Statistically Significant)

[AIDev v5 (Agentic) - Sanitized (External Friction)]
  Paired 2x2 Table: [Both Silent (a): 248, Treat Only (b): 48, Ctrl Only (c): 76, Both Reviewed (d): 315]
  - Matched Odds Ratio (OR): 0.6316 (95% CI: [0.4400, 0.9065])
  - Exact McNemar p-value   : 1.4985e-02 (Statistically Significant)

[MOSAIC-3M (Agentic Only) - Raw (Total Friction)]
  Paired 2x2 Table: [Both Silent (a): 22, Treat Only (b): 18, Ctrl Only (c): 11, Both Reviewed (d): 29]
  - Matched Odds Ratio (OR): 1.6364 (95% CI: [0.7729, 3.4645])
  - Exact McNemar p-value   : 2.6493e-01 (Not Significant)

[MOSAIC-3M (Agentic Only) - Sanitized (External Friction)]
  Paired 2x2 Table: [Both

## Section 5: Governance Moderation Analysis
### 5.1 Pooled Interaction Logistic Regression Model

To address the baseline equivalence limitation of comparing treatment deltas across separately matched cohorts with different baseline covariate distributions, we model the probability of external human review intervention ($\Pr(\text{is\_externally\_reviewed} = 1)$) using a parametric logistic regression model on the complete unmatched cohort ($N=3,648$). 

We evaluate the interaction term between agent identity and dual-scope infrastructure treatment ($\text{is\_agent} \times \text{is\_treatment}$), controlling for file scope (`log_app_count`, `log_app_line_churn`), test inclusion (`has_tests`), repository popularity (`log_stars`), and organization privilege (`is_core_member`).

In [17]:
# ==============================================================================
# 6.5 INTERACTION EFFECT LOGISTIC REGRESSION (MODERATION ANALYSIS)
# ==============================================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm

print("=== 5.1 Pooled Interaction Logistic Regression Model ===")

# --------------------------------------------------------------------------
# Task 1: Data Preparation
# --------------------------------------------------------------------------
# 1. Left-merge sanitized metrics from df_eval_mosaic onto df_mosaic_pr_filtered using 'id'
df_model = df_mosaic_pr_filtered.merge(
    df_eval_mosaic[["id", "external_comments", "external_review_iterations"]],
    on="id",
    how="left"
)
df_model["external_review_iterations"] = df_model["external_review_iterations"].fillna(0)
df_model["external_comments"] = df_model["external_comments"].fillna(0)

# 2. Filter dataset to include ONLY scope == 'App_Only' and scope == 'App_and_Infra'
df_model = df_model[df_model["scope"].isin(["App_Only", "App_and_Infra"])].copy()

# 3. Binary outcome variable: is_externally_reviewed (1 if external comments > 0 or external iterations > 0, else 0)
df_model["is_externally_reviewed"] = (
    (df_model["external_comments"] > 0) | (df_model["external_review_iterations"] > 0)
).astype(int)

# 4. Binary treatment variable: is_treatment (1 if scope == 'App_and_Infra', else 0)
df_model["is_treatment"] = (df_model["scope"] == "App_and_Infra").astype(int)

# 5. Binary identity variable: is_agent (1 if agent_source != 'Human', else 0)
df_model["is_agent"] = (df_model["agent_source"] != "Human").astype(int)

# 6. Interaction term: is_agent_x_treatment (is_agent * is_treatment)
df_model["is_agent_x_treatment"] = df_model["is_agent"] * df_model["is_treatment"]

# 7. Binary privilege control: is_core_member (1 if author_association in OWNER, MEMBER, COLLABORATOR, else 0)
df_model["is_core_member"] = df_model["author_association"].isin(["OWNER", "MEMBER", "COLLABORATOR"]).astype(int)

# 8. Ensure has_tests is converted to integer (1 or 0)
df_model["has_tests"] = df_model["has_tests"].astype(int)

# Ensure log-transformed covariates exist
if "log_app_count" not in df_model.columns:
    df_model["log_app_count"] = np.log(df_model["app_count"])
if "log_app_line_churn" not in df_model.columns:
    df_model["log_app_line_churn"] = np.log(df_model["app_line_churn"] + 1)
if "log_stars" not in df_model.columns:
    df_model["log_stars"] = np.log(df_model["stars"] + 1)

# --------------------------------------------------------------------------
# Task 2: Logistic Regression Modeling
# --------------------------------------------------------------------------
modeling_cols = [
    "is_externally_reviewed",
    "is_treatment", "is_agent", "is_agent_x_treatment",
    "log_app_count", "log_app_line_churn", "has_tests", "log_stars", "is_core_member"
]

clean_df = df_model[modeling_cols].dropna().copy()
print(f"Sample Size (Complete Cases): {len(clean_df):,} PRs (Agent: {clean_df['is_agent'].sum():,}, Human: {(clean_df['is_agent'] == 0).sum():,})")

x_cols = ["is_treatment", "is_agent", "is_agent_x_treatment", "log_app_count", "log_app_line_churn", "has_tests", "log_stars", "is_core_member"]
X = sm.add_constant(clean_df[x_cols])
y = clean_df["is_externally_reviewed"]

logit_model = sm.Logit(y, X).fit(disp=False)

print("\n" + "=" * 70)
print("  LOGIT REGRESSION RESULTS (MODERATION ANALYSIS)")
print("=" * 70)
print(logit_model.summary().as_text())

# Exponentiated Odds Ratios & 95% Confidence Intervals
or_table = pd.DataFrame({
    "coef": logit_model.params,
    "Odds Ratio (exp(coef))": np.exp(logit_model.params),
    "se(coef)": logit_model.bse,
    "z": logit_model.tvalues,
    "p-value": logit_model.pvalues,
    "95% CI Lower": np.exp(logit_model.conf_int()[0]),
    "95% CI Upper": np.exp(logit_model.conf_int()[1])
})

print("\n" + "=" * 70)
print("  EXPONENTIATED ODDS RATIOS & 95% CONFIDENCE INTERVALS")
print("=" * 70)
print(or_table.round(4).to_string())

# Key Interaction Summary
inter_coef = logit_model.params["is_agent_x_treatment"]
inter_or = np.exp(inter_coef)
inter_p = logit_model.pvalues["is_agent_x_treatment"]
inter_ci_l = np.exp(logit_model.conf_int().loc["is_agent_x_treatment", 0])
inter_ci_u = np.exp(logit_model.conf_int().loc["is_agent_x_treatment", 1])

print("\n--- Interaction Effect Summary (is_agent x is_treatment) ---")
print(f" * Interaction Coefficient : {inter_coef:+.4f} (SE: {logit_model.bse['is_agent_x_treatment']:.4f}, z={logit_model.tvalues['is_agent_x_treatment']:.3f})")
print(f" * Interaction Odds Ratio  : {inter_or:.4f} (95% CI: [{inter_ci_l:.4f}, {inter_ci_u:.4f}])")
signif_label = "Statistically Significant" if inter_p < 0.05 else "Not Significant (p >= 0.05)"
print(f" * Statistical Significance: p={inter_p:.4f} ({signif_label})")

=== 5.1 Pooled Interaction Logistic Regression Model ===
Sample Size (Complete Cases): 3,648 PRs (Agent: 1,920, Human: 1,728)

  LOGIT REGRESSION RESULTS (MODERATION ANALYSIS)
                             Logit Regression Results                             
Dep. Variable:     is_externally_reviewed   No. Observations:                 3648
Model:                              Logit   Df Residuals:                     3639
Method:                               MLE   Df Model:                            8
Date:                    Fri, 25 Sep 2026   Pseudo R-squ.:                  0.1721
Time:                            15:07:27   Log-Likelihood:                -1986.3
converged:                           True   LL-Null:                       -2399.2
Covariance Type:                nonrobust   LLR p-value:                5.826e-173
                           coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------

### 5.2 Governance-Stratified Moderation (External Contributors vs. Maintainers)

In Section 5.1, the pooled moderation model yielded an interaction term (`is_agent_x_treatment`) of $p = 0.1235$, while repository privilege (`is_core_member`) was overwhelmingly significant ($z = -15.05, p < 10^{-50}$). To determine whether the review bypass on infrastructure additions is an artifact of maintainer permissions or a true agent phenomenon that persists among external contributors, we execute a Governance-Stratified Moderation Analysis.

We stratify the complete unmatched cohort ($N = 3,648$) into two distinct governance tiers:
1. **Stratum A (External Contributors)**: `is_core_member == 0` (`CONTRIBUTOR` / `NONE`, $N = 2,232$).
2. **Stratum B (Core Maintainers)**: `is_core_member == 1` (`OWNER` / `MEMBER` / `COLLABORATOR`, $N = 1,416$).

We estimate separate logistic regression models on each stratum predicting $\Pr(\text{is\_externally\_reviewed} = 1)$, omitting `is_core_member` to avoid singular matrices, and directly contrast the interaction coefficients against the pooled baseline.

In [18]:
# ==============================================================================
# 6.6 GOVERNANCE-STRATIFIED MODERATION ANALYSIS
# ==============================================================================
print("=== 5.2 Governance-Stratified Moderation Analysis (External vs. Core) ===")

# --------------------------------------------------------------------------
# 1. Data Slicing (Complete Cases)
# --------------------------------------------------------------------------
df_ext = clean_df[clean_df["is_core_member"] == 0].copy()
df_core = clean_df[clean_df["is_core_member"] == 1].copy()

print("\n--- Stratum Sample Sizes & Descriptive Event Counts ---")
print("Stratum A (External Contributors - CONTRIBUTOR / NONE):")
print(f"  * Total PRs: {len(df_ext):,} (Agent: {df_ext['is_agent'].sum():,}, Human: {(df_ext['is_agent'] == 0).sum():,})")
print(f"  * External Review Rate: {df_ext['is_externally_reviewed'].mean() * 100:.2f}% ({df_ext['is_externally_reviewed'].sum():,} / {len(df_ext):,})")

print("\nStratum B (Core Maintainers - OWNER / MEMBER / COLLABORATOR):")
print(f"  * Total PRs: {len(df_core):,} (Agent: {df_core['is_agent'].sum():,}, Human: {(df_core['is_agent'] == 0).sum():,})")
print(f"  * External Review Rate: {df_core['is_externally_reviewed'].mean() * 100:.2f}% ({df_core['is_externally_reviewed'].sum():,} / {len(df_core):,})")

# --------------------------------------------------------------------------
# 2. Model Estimation (statsmodels.api.Logit)
# --------------------------------------------------------------------------
x_cols_strat = ["is_treatment", "is_agent", "is_agent_x_treatment", "log_app_count", "log_app_line_churn", "has_tests", "log_stars"]

# Stratum A: External Contributors
X_ext = sm.add_constant(df_ext[x_cols_strat])
y_ext = df_ext["is_externally_reviewed"]
model_ext = sm.Logit(y_ext, X_ext).fit(disp=False)

# Stratum B: Core Maintainers
X_core = sm.add_constant(df_core[x_cols_strat])
y_core = df_core["is_externally_reviewed"]
model_core = sm.Logit(y_core, X_core).fit(disp=False)

def build_or_table(model):
    conf = model.conf_int()
    return pd.DataFrame({
        "coef": model.params,
        "Odds Ratio (exp(coef))": np.exp(model.params),
        "se(coef)": model.bse,
        "z": model.tvalues,
        "p-value": model.pvalues,
        "95% CI Lower": np.exp(conf.iloc[:, 0]),
        "95% CI Upper": np.exp(conf.iloc[:, 1])
    }).round(4)

print("\n" + "=" * 75)
print("  STRATUM A: EXTERNAL CONTRIBUTORS (CONTRIBUTOR / NONE) - LOGIT RESULTS")
print("=" * 75)
print(model_ext.summary().as_text())
print("\n--- Stratum A: Exponentiated Odds Ratios & 95% Confidence Intervals ---")
print(build_or_table(model_ext).to_string())

print("\n" + "=" * 75)
print("  STRATUM B: CORE MAINTAINERS (OWNER / MEMBER / COLLAB) - LOGIT RESULTS")
print("=" * 75)
print(model_core.summary().as_text())
print("\n--- Stratum B: Exponentiated Odds Ratios & 95% Confidence Intervals ---")
print(build_or_table(model_core).to_string())

# --------------------------------------------------------------------------
# 3. Direct Contrast Table (Comparison of is_agent_x_treatment Across Models)
# --------------------------------------------------------------------------
def extract_interaction_row(cohort_name, model, n_obs):
    coef = model.params["is_agent_x_treatment"]
    se = model.bse["is_agent_x_treatment"]
    odds_ratio = np.exp(coef)
    conf = model.conf_int()
    ci_lower = np.exp(conf.loc["is_agent_x_treatment"].iloc[0])
    ci_upper = np.exp(conf.loc["is_agent_x_treatment"].iloc[1])
    z_val = model.tvalues["is_agent_x_treatment"]
    p_val = model.pvalues["is_agent_x_treatment"]
    return {
        "Model Cohort": cohort_name,
        "N": f"{n_obs:,}",
        "Interaction Coef": f"{coef:+.4f}",
        "SE": f"{se:.4f}",
        "Odds Ratio": f"{odds_ratio:.4f}",
        "95% CI": f"[{ci_lower:.4f}, {ci_upper:.4f}]",
        "z-value": f"{z_val:.3f}",
        "p-value": f"{p_val:.4f}"
    }

contrast_rows = [
    extract_interaction_row("Pooled Model (Section 6.5)", logit_model, len(clean_df)),
    extract_interaction_row("Stratum A: External Contributors (CONTRIBUTOR / NONE)", model_ext, len(df_ext)),
    extract_interaction_row("Stratum B: Core Maintainers (OWNER / MEMBER / COLLAB)", model_core, len(df_core)),
]

df_contrast = pd.DataFrame(contrast_rows)
print("\n" + "=" * 90)
print("  DIRECT CONTRAST TABLE: INTERACTION EFFECT (is_agent_x_treatment) ACROSS STRATA")
print("=" * 90)
print(df_contrast.to_string(index=False))


=== 5.2 Governance-Stratified Moderation Analysis (External vs. Core) ===

--- Stratum Sample Sizes & Descriptive Event Counts ---
Stratum A (External Contributors - CONTRIBUTOR / NONE):
  * Total PRs: 2,232 (Agent: 1,320, Human: 912)
  * External Review Rate: 71.42% (1,594 / 2,232)

Stratum B (Core Maintainers - OWNER / MEMBER / COLLABORATOR):
  * Total PRs: 1,416 (Agent: 600, Human: 816)
  * External Review Rate: 50.35% (713 / 1,416)

  STRATUM A: EXTERNAL CONTRIBUTORS (CONTRIBUTOR / NONE) - LOGIT RESULTS
                             Logit Regression Results                             
Dep. Variable:     is_externally_reviewed   No. Observations:                 2232
Model:                              Logit   Df Residuals:                     2224
Method:                               MLE   Df Model:                            7
Date:                    Fri, 25 Sep 2026   Pseudo R-squ.:                 0.09929
Time:                            15:07:27   Log-Likelihood:             

## Section 6: Contextual Review Bottlenecks and Review Latency Dynamics
### 6.1 Infrastructure Subdomain Review Friction (CI/CD vs. Automation)

To determine which operational domains generate the highest human review overhead, we decompose dual-scope (`App_and_Infra`) PRs by their infrastructure subcategories: `CI_CD`, `Automation`, `Containers`, and `Provisioning`.

We evaluate both **Raw** and **Sanitized (Author Self-Comment Purged)** metrics across cohorts to identify which infrastructure types trigger the most intensive human review iterations and discussion comments.

In [19]:
print("=== 6.1 Infrastructure Subdomain Review Friction (CI/CD vs. Automation) ===")

# --------------------------------------------------------------------------
# Step 1: AIDev v5 Bottlenecks
# --------------------------------------------------------------------------
# Extract the ids of the Treatment PRs from df_aidev_matched (where match_role == 'Treatment')
treat_ids_aidev = set(df_aidev_matched[df_aidev_matched['match_role'] == 'Treatment']['id'])

# Load pr_commit_details.parquet using load_dataset_cached and filter to Treatment PR ids
df_commit_details_b = load_dataset_cached(
    hf_url="hf://datasets/hao-li/AIDev-7.6M/pr_commit_details.parquet",
    filename="aidev_v5_pr_commit_details.parquet",
    columns=["pr_id", "filename"]
)
df_commit_treat = df_commit_details_b[df_commit_details_b['pr_id'].isin(treat_ids_aidev)].dropna(subset=['filename']).copy()

# Apply classify_file_domain(filename) to filename column to extract infra_category
df_commit_treat['infra_category'] = df_commit_treat['filename'].apply(
    lambda fn: classify_file_domain(fn).get('infra_category')
)

# Filter out rows where infra_category is None/NaN, then drop duplicates to get unique (pr_id, infra_category)
df_pr_infra_aidev = df_commit_treat[df_commit_treat['infra_category'].notna()][['pr_id', 'infra_category']].drop_duplicates()

# Merge directly with matched Treatment PR table
df_pr_infra_aidev = df_pr_infra_aidev.merge(
    df_aidev_matched[['id', 'human_review_iterations', 'human_review_comments', 'external_review_iterations', 'external_comments']],
    left_on='pr_id',
    right_on='id',
    how='left'
)

# Group by infra_category and calculate count of PRs, mean/median raw and sanitized metrics
aidev_bottlenecks = df_pr_infra_aidev.groupby('infra_category').agg(
    PR_Count=('id', 'count'),
    Mean_Raw_Iter=('human_review_iterations', 'mean'),
    Median_Raw_Iter=('human_review_iterations', 'median'),
    Mean_San_Iter=('external_review_iterations', 'mean'),
    Median_San_Iter=('external_review_iterations', 'median'),
    Mean_Raw_Comm=('human_review_comments', 'mean'),
    Median_Raw_Comm=('human_review_comments', 'median'),
    Mean_San_Comm=('external_comments', 'mean'),
    Median_San_Comm=('external_comments', 'median')
).reset_index().sort_values(by='PR_Count', ascending=False)

# --------------------------------------------------------------------------
# Step 2: MOSAIC-3M Bottlenecks
# --------------------------------------------------------------------------
# Extract Treatment PRs from df_mosaic_matched (where match_role == 'Treatment')
df_mosaic_treat = df_mosaic_matched[df_mosaic_matched['match_role'] == 'Treatment'][['id']].drop_duplicates()

# Merge this with df_mosaic_pr_filtered[['id', 'files']] to get raw files array
df_mosaic_treat = df_mosaic_treat.merge(df_mosaic_pr_filtered[['id', 'files']], on='id', how='inner')

# Iterate through files array for these PRs, apply classify_file_domain, and extract unique infra_category per PR
mosaic_infra_rows = []
for _, row in df_mosaic_treat.iterrows():
    pr_id = row['id']
    files_val = row['files']
    if isinstance(files_val, (list, np.ndarray)):
        seen_cats = set()
        for f_item in files_val:
            path = f_item.get('path') if isinstance(f_item, dict) else str(f_item)
            if path:
                res = classify_file_domain(path)
                cat = res.get('infra_category')
                if cat and cat not in seen_cats:
                    seen_cats.add(cat)
                    mosaic_infra_rows.append({'id': pr_id, 'infra_category': cat})

# Create a dataframe of (id, infra_category)
df_pr_infra_mosaic = pd.DataFrame(mosaic_infra_rows)

# Merge directly with matched Treatment PR table
df_pr_infra_mosaic = df_pr_infra_mosaic.merge(
    df_mosaic_matched[['id', 'human_review_iterations', 'human_review_comments', 'external_review_iterations', 'external_comments']],
    on='id',
    how='left'
)

# Group by infra_category and calculate count of PRs, mean/median raw and sanitized metrics
mosaic_bottlenecks = df_pr_infra_mosaic.groupby('infra_category').agg(
    PR_Count=('id', 'count'),
    Mean_Raw_Iter=('human_review_iterations', 'mean'),
    Median_Raw_Iter=('human_review_iterations', 'median'),
    Mean_San_Iter=('external_review_iterations', 'mean'),
    Median_San_Iter=('external_review_iterations', 'median'),
    Mean_Raw_Comm=('human_review_comments', 'mean'),
    Median_Raw_Comm=('human_review_comments', 'median'),
    Mean_San_Comm=('external_comments', 'mean'),
    Median_San_Comm=('external_comments', 'median')
).reset_index().sort_values(by='PR_Count', ascending=False)

# --------------------------------------------------------------------------
# Step 3: Output
# --------------------------------------------------------------------------
print("\n--- AIDev v5: Infrastructure Category Bottlenecks ---")
print(aidev_bottlenecks.round(2).to_string(index=False))

print("\n--- MOSAIC-3M (Agentic Only): Infrastructure Category Bottlenecks ---")
print(mosaic_bottlenecks.round(2).to_string(index=False))


=== 6.1 Infrastructure Subdomain Review Friction (CI/CD vs. Automation) ===



--- AIDev v5: Infrastructure Category Bottlenecks ---
infra_category  PR_Count  Mean_Raw_Iter  Median_Raw_Iter  Mean_San_Iter  Median_San_Iter  Mean_Raw_Comm  Median_Raw_Comm  Mean_San_Comm  Median_San_Comm
         CI_CD       532           2.45              1.0           2.22              1.0           3.28              1.0           2.93              0.0
    Automation       109           3.33              2.0           3.11              1.0           4.66              2.0           4.24              1.0
    Containers       108           1.68              0.0           1.50              0.0           2.50              0.0           2.29              0.0
  Provisioning        61           2.07              1.0           1.95              1.0           2.80              0.0           2.62              0.0

--- MOSAIC-3M (Agentic Only): Infrastructure Category Bottlenecks ---
infra_category  PR_Count  Mean_Raw_Iter  Median_Raw_Iter  Mean_San_Iter  Median_San_Iter  Mean_Raw_Comm  Medi

### 6.2 Latency Dynamics of Human Review Oversight vs. Silent Integration

To empirically evaluate how human review oversight drives latency inflation, we examine how **Silent Integration** status (`is_silent_integration`: exactly 0 comments and 0 iterations) interacts with Treatment (`App_and_Infra`) versus Control (`App_Only`) groups across our matched cohorts.

We compare the median and mean merge latencies for explicitly reviewed PRs versus silently integrated PRs to determine whether human review oversight—rather than background automated testing—is the primary driver of merge latency delays.

In [20]:
print("=== 6.2 Latency Dynamics of Human Review Oversight vs. Silent Integration ===")

def evaluate_matched_silent_integration(df_matched, df_pr_filtered, dataset_name):
    # 1. Merge matched cohort with filtered PR table on id
    df_eval = df_matched[['id', 'match_role', 'human_review_comments', 'human_review_iterations']].merge(
        df_pr_filtered[['id', 'merged_at', 'created_at']],
        on='id',
        how='left'
    )
    
    # 2. Convert timestamps to datetime
    merged_dt = pd.to_datetime(df_eval['merged_at'], utc=True, errors='coerce')
    created_dt = pd.to_datetime(df_eval['created_at'], utc=True, errors='coerce')
    
    # 3. Calculate merge_latency in hours
    df_eval['merge_latency'] = (merged_dt - created_dt).dt.total_seconds() / 3600.0
    
    # 4. Filter clock-skew anomalies
    df_eval = df_eval[df_eval['merge_latency'] >= 0].copy()
    
    # 5. Define is_silent_integration boolean column
    df_eval['is_silent_integration'] = (
        (df_eval['human_review_comments'] == 0) & 
        (df_eval['human_review_iterations'] == 0)
    )
    
    # 6. Group by match_role and is_silent_integration and aggregate latency metrics
    summary_df = df_eval.groupby(['match_role', 'is_silent_integration']).agg(
        PR_Count=('id', 'count'),
        Median_Latency_Hours=('merge_latency', 'median'),
        Mean_Latency_Hours=('merge_latency', 'mean')
    ).reset_index()
    
    # 7. Calculate % of Match Role
    summary_df['% of Match Role'] = (
        summary_df['PR_Count'] / 
        summary_df.groupby('match_role')['PR_Count'].transform('sum') * 100
    ).round(2)
    
    # 8. Print dataset name and display formatted table
    print(f"\n--- {dataset_name}: Silent Integration vs. Explicitly Reviewed Latency Dynamics ---")
    display(summary_df.round(2))
    return summary_df

# Run helper function for both matched datasets
evaluate_matched_silent_integration(df_aidev_matched, df_aidev_pr_filtered, "AIDev v5 Matched Cohort")
evaluate_matched_silent_integration(df_mosaic_matched, df_mosaic_pr_filtered, "MOSAIC-3M Matched Cohort")

=== 6.2 Latency Dynamics of Human Review Oversight vs. Silent Integration ===

--- AIDev v5 Matched Cohort: Silent Integration vs. Explicitly Reviewed Latency Dynamics ---


,match_role,is_silent_integration,PR_Count,Median_Latency_Hours,Mean_Latency_Hours,% of Match Role
0,Control,False,388,22.38,85.73,56.48
1,Control,True,299,0.10,6.18,43.52
2,Treatment,False,426,49.80,164.33,62.01
3,Treatment,True,261,0.43,23.49,37.99



--- MOSAIC-3M Matched Cohort: Silent Integration vs. Explicitly Reviewed Latency Dynamics ---


,match_role,is_silent_integration,PR_Count,Median_Latency_Hours,Mean_Latency_Hours,% of Match Role
0,Control,False,47,7.20,139.42,58.75
1,Control,True,33,0.68,6.74,41.25
2,Treatment,False,40,15.59,93.03,50.00
3,Treatment,True,40,0.51,4.39,50.00


,match_role,is_silent_integration,PR_Count,Median_Latency_Hours,Mean_Latency_Hours,% of Match Role
0,Control,False,47,7.201944,139.416070,58.75
1,Control,True,33,0.679444,6.741035,41.25
2,Treatment,False,40,15.587083,93.025847,50.00
3,Treatment,True,40,0.514722,4.392750,50.00
